# Amazon ML Challenge — Business Entity Resolution
## Kaggle-safe 10K development notebook

This notebook is based on the supplied WIP package and preserves its validated blocking logic and
measured 10K results. It is designed to run top-to-bottom without network/package installation.

**Validated WIP measurements (source-derived):**
- S2 candidate recall: 98.62%
- S3 candidate recall: 98.26%
- Combined candidate recall: 98.43%
- Validation macro F0.5: 0.9663
- 10K end-to-end development runtime reported by the WIP: 243.3 s

**Resource policy:** default S1 is capped at 10,000 and TEST generation is OFF. This prevents an
accidental multi-million-row run from consuming hours or filling a Kaggle working disk. The actual
competition TEST outputs must only be generated after the real TEST files are available and their
size has been checked.


## 1. Imports and dependency checks

In [1]:

import os, re, sys, gc, json, time, shutil, zipfile, unicodedata, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RUN_START = time.time()

# ---- RapidFuzz: fast C++ string similarity. Standard on Kaggle. ----
try:
    from rapidfuzz import fuzz as rf_fuzz
    from rapidfuzz import distance as rf_distance
    HAVE_RAPIDFUZZ = True
except Exception:
    HAVE_RAPIDFUZZ = False

# ---- LightGBM: MIT-licensed, compact GBDT well under the 8B-parameter cap. ----
try:
    import lightgbm as lgb
    HAVE_LIGHTGBM = True
except Exception:
    HAVE_LIGHTGBM = False

# ---- scikit-learn fallback model (always available on Kaggle). ----
from sklearn.ensemble import HistGradientBoostingClassifier

# ---- psutil for RAM reporting; degrade gracefully if unavailable. ----
try:
    import psutil
    HAVE_PSUTIL = True
except Exception:
    HAVE_PSUTIL = False

print(f"RapidFuzz available : {HAVE_RAPIDFUZZ}")
print(f"LightGBM available  : {HAVE_LIGHTGBM}")
print(f"psutil available    : {HAVE_PSUTIL}")

if not HAVE_RAPIDFUZZ:
    print("WARNING: RapidFuzz not found. Falling back to a vectorizable pure-Python "
          "character-trigram / token Jaccard feature set (NOT the slow difflib.SequenceMatcher "
          "used by the old reference implementation). This keeps the 10K dev run practical, "
          "but installing rapidfuzz (`pip install rapidfuzz`) is strongly recommended before "
          "scaling to the full dataset.")

if not HAVE_LIGHTGBM:
    print("WARNING: LightGBM not found. Falling back to sklearn HistGradientBoostingClassifier "
          "(also a compact, license-friendly GBDT well under the competition's 8B-parameter cap).")


RapidFuzz available : False
LightGBM available  : True
psutil available    : True


## 2. Dataset discovery
Searches Kaggle's `/kaggle/input` mounts first, then falls back to a local `dataset/` layout so the same notebook runs unchanged on Kaggle or on a local machine. No Windows-specific paths are hard-coded anywhere.

In [2]:

def find_file(filename, search_roots):
    """Search each root recursively for filename; return the first match or None."""
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        # Fast path: direct child.
        direct = root / filename
        if direct.exists():
            return str(direct)
        # Recursive search (Kaggle datasets are often nested one level deeper).
        for p in root.rglob(filename):
            return str(p)
    return None


SEARCH_ROOTS = [
    "/kaggle/input",
    "/kaggle/working",
    "/mnt/user-data/uploads",   # this conversation's uploaded files
    "dataset/train",
    "dataset",
    ".",
]

TRAIN_FILES = {
    "s1": "train_source1.tsv",
    "s2": "train_source2.tsv",
    "s3": "train_source3.tsv",
    "gt": "train_ground_truth.tsv",
}

DATA_PATHS = {k: find_file(v, SEARCH_ROOTS) for k, v in TRAIN_FILES.items()}

print("Discovered TRAIN files:")
for k, v in DATA_PATHS.items():
    print(f"  {k:>3}: {v}")

missing = [k for k, v in DATA_PATHS.items() if v is None]
if missing:
    raise FileNotFoundError(
        f"Could not locate required train file(s): {missing}. "
        f"Attach the competition dataset (or this conversation's 10K upload) as a Kaggle "
        f"Notebook input and re-run this cell."
    )


Discovered TRAIN files:
   s1: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source1.tsv
   s2: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source2.tsv
   s3: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_source3.tsv
   gt: /kaggle/input/datasets/tejasweerajput/amazon-ml-challenge-2026-student-resource/student_resource/dataset/train/train_ground_truth.tsv


## 3. Configuration and Kaggle resource policy

"
This notebook is a **10K validated development / smoke-test notebook**. The default is intentionally bounded because the older WIP full-scale switch can create hundreds of millions of candidate/feature operations and exceed a ~20 GB Kaggle working-disk budget. The validated 10K measurements from the WIP package are preserved below; do not infer full-scale runtime from them.


In [3]:
# ============================================================
# CONFIGURATION — KAGGLE-SAFE DEFAULTS
# ============================================================
# IMPORTANT:
# This notebook is intentionally locked to the validated 10K development run.
# The WIP benchmark was measured on:
#   S1=10,000, S2=100,000, S3=100,000
#   Combined candidate recall = 98.43%
#   Macro F0.5 = 0.9663
#
# DO NOT set S1_LIMIT=None in this notebook. That switches the old WIP
# implementation into a multi-million-row experiment and can run for hours
# and/or exceed Kaggle's ~20 GB working-disk budget.
S1_LIMIT = 10_000
S1_CHUNK_SIZE = 2_500

# Keep the expensive per-blocker diagnostic OFF. It is not required for
# submission and was only used during development.
RUN_BLOCKER_DIAGNOSTICS = False

# TEST generation is OFF by default so a notebook run never accidentally
# launches a multi-million-row TEST job. Turn this on only after inspecting
# TEST sizes and confirming the resource budget.
RUN_TEST = False
TEST_S1_LIMIT = 10_000  # safe smoke/development limit; use None only deliberately

# Frequency caps from the validated WIP/reference blocker implementation.
DEFAULT_FREQ_CAPS = {
    "NP6": 300, "NP8": 300, "NS6": 300, "FL": 400, "FL3": 400, "F2": 400, "FML": 400,
    "AN": 400, "A2": 400, "AW2": 400, "AL2": 80, "NL": 400,
    "A4W": 10, "AT2": 20, "RAT20": 20,
    "UNP4": 300, "UNP6": 300, "UFL": 300,
}
DEFAULT_FREQ_CAP_FALLBACK = 500

OUTPUT_DIR = Path("output")
REPORTS_DIR = Path("reports")
OUTPUT_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
VALIDATION_FRACTION = 0.20
TEAM_NAME = "team"

print("Kaggle-safe configuration")
print("S1_LIMIT            :", S1_LIMIT)
print("S1_CHUNK_SIZE       :", S1_CHUNK_SIZE)
print("RUN_BLOCKER_DIAGNOSTICS:", RUN_BLOCKER_DIAGNOSTICS)
print("RUN_TEST            :", RUN_TEST)
print("TEST_S1_LIMIT       :", TEST_S1_LIMIT)


Kaggle-safe configuration
S1_LIMIT            : 10000
S1_CHUNK_SIZE       : 2500
RUN_BLOCKER_DIAGNOSTICS: False
RUN_TEST            : False
TEST_S1_LIMIT       : 10000


## 4. Dataset statistics
## 5. Resource monitoring helpers

In [4]:
def resource_snapshot(label=""):
    """Report RAM and disk usage for the working directory."""
    lines = [f"--- resource snapshot: {label} ---"]

    if HAVE_PSUTIL:
        proc = psutil.Process(os.getpid())
        rss_mb = proc.memory_info().rss / (1024 ** 2)
        vm = psutil.virtual_memory()

        lines.append(f"  process RSS      : {rss_mb:,.1f} MB")
        lines.append(
            f"  system used/total: "
            f"{vm.used/1024**3:,.2f} / {vm.total/1024**3:,.2f} GB"
        )
    else:
        lines.append("  (psutil unavailable — RAM not reported)")

    total, used, free = shutil.disk_usage(".")
    lines.append(
        f"  disk used/total  : "
        f"{used/1024**3:,.2f} / {total/1024**3:,.2f} GB"
    )
    lines.append(
        f"  disk free        : "
        f"{free/1024**3:,.2f} GB"
    )
    lines.append(
        f"  elapsed since start: "
        f"{time.time() - RUN_START:,.1f}s"
    )

    text = "\n".join(lines)
    print(text)
    return text


def read_tsv(path, limit=None):
    """
    Robust TSV reader.

    - Explicit tab separator
    - IDs remain strings
    - Does not interpret empty strings as NaN
    """
    return pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        nrows=limit,
        engine="c"
    )


def load_ground_truth_for_s1(gt_path, s1_ids, chunk_size=100_000):
    """
    Load ONLY GT rows whose source1_entity_id belongs to the loaded S1 subset.

    IMPORTANT:
    The full GT file can contain ~2.2M rows. We must NOT load the entire
    file into memory just to evaluate the first 10K S1 records.

    The file is scanned in chunks and matching rows are retained.
    """
    s1_id_set = set(s1_ids)

    if not s1_id_set:
        return pd.DataFrame(
            columns=["source1_entity_id", "matched_entity_ids"]
        )

    collected = []
    rows_scanned = 0
    chunks_seen = 0

    print("Loading only GT rows required for the current S1 subset...")
    print(f"Target S1 IDs: {len(s1_id_set):,}")
    print(f"GT chunk size: {chunk_size:,}")

    for chunk in pd.read_csv(
        gt_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        chunksize=chunk_size,
        engine="c"
    ):
        chunks_seen += 1
        rows_scanned += len(chunk)

        # Keep only GT rows belonging to our loaded S1 subset
        mask = chunk["source1_entity_id"].isin(s1_id_set)

        if mask.any():
            matched = chunk.loc[
                mask,
                ["source1_entity_id", "matched_entity_ids"]
            ].copy()

            collected.append(matched)

        # We cannot safely stop merely because a chunk had matches,
        # because GT may not be sorted by source1_entity_id.
        #
        # We can stop only when every requested S1 ID has been found.
        if collected:
            found_ids = set(
                pd.concat(
                    collected,
                    ignore_index=True
                )["source1_entity_id"]
            )

            if len(found_ids) >= len(s1_id_set):
                print(
                    f"All {len(s1_id_set):,} requested S1 IDs "
                    f"found after scanning {rows_scanned:,} GT rows."
                )
                break

        if chunks_seen % 10 == 0:
            print(
                f"  scanned {rows_scanned:,} GT rows..."
            )

    if not collected:
        print("WARNING: No matching GT rows found.")
        return pd.DataFrame(
            columns=["source1_entity_id", "matched_entity_ids"]
        )

    gt = pd.concat(
        collected,
        ignore_index=True
    )

    # Defensive cleanup
    gt = gt.drop_duplicates(
        subset=["source1_entity_id"],
        keep="first"
    ).reset_index(drop=True)

    print(
        f"GT rows retained: {len(gt):,}"
    )

    return gt


# ============================================================
# SAFETY CHECKS
# ============================================================

if S1_LIMIT is None or S1_LIMIT > 10_000:
    raise RuntimeError(
        "KAGGLE-SAFE STOP: S1_LIMIT must be <= 10,000 "
        "in this development notebook."
    )


# ============================================================
# LOAD DATA
# ============================================================

t_load = time.time()

print("=" * 60)
print("LOADING KAGGLE 10K DEVELOPMENT DATA")
print("=" * 60)

# S1: intentionally limited to 10K
S1 = read_tsv(
    DATA_PATHS["s1"],
    limit=S1_LIMIT
)

print(f"S1 loaded: {len(S1):,}")

# S2 and S3 are the 100K development/reference datasets
S2 = read_tsv(
    DATA_PATHS["s2"]
)

print(f"S2 loaded: {len(S2):,}")

S3 = read_tsv(
    DATA_PATHS["s3"]
)

print(f"S3 loaded: {len(S3):,}")


# ============================================================
# BASIC COLUMN VALIDATION
# ============================================================

REQUIRED_COLUMNS = {
    "entity_id",
    "business_name",
    "business_address",
    "country"
}

for name, df in [
    ("S1", S1),
    ("S2", S2),
    ("S3", S3)
]:
    missing = REQUIRED_COLUMNS - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {sorted(missing)}"
        )


# ============================================================
# LOAD ONLY REQUIRED GROUND-TRUTH ROWS
# ============================================================

GT_RAW = load_ground_truth_for_s1(
    DATA_PATHS["gt"],
    S1["entity_id"].astype(str).tolist(),
    chunk_size=100_000
)


# ============================================================
# FINAL DATASET STATISTICS
# ============================================================

print()
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print(f"S1 rows            : {len(S1):,}")
print(f"S2 rows            : {len(S2):,}")
print(f"S3 rows            : {len(S3):,}")
print(f"Ground-truth rows  : {len(GT_RAW):,}")

print()
print("S1 country distribution:")
print(S1["country"].value_counts())

print()
print("S2 country distribution:")
print(S2["country"].value_counts())

print()
print("S3 country distribution:")
print(S3["country"].value_counts())

print()

resource_snapshot("after data load")

print()
print(
    f"Total loading time: "
    f"{time.time() - t_load:,.1f}s"
)

print()
print("=" * 60)
print("DATA LOAD COMPLETE")
print("=" * 60)

LOADING KAGGLE 10K DEVELOPMENT DATA
S1 loaded: 10,000
S2 loaded: 5,034,616
S3 loaded: 5,285,603
Loading only GT rows required for the current S1 subset...
Target S1 IDs: 10,000
GT chunk size: 100,000
  scanned 1,000,000 GT rows...
  scanned 2,000,000 GT rows...
All 10,000 requested S1 IDs found after scanning 2,206,821 GT rows.
GT rows retained: 10,000

DATASET STATISTICS
S1 rows            : 10,000
S2 rows            : 5,034,616
S3 rows            : 5,285,603
Ground-truth rows  : 10,000

S1 country distribution:
country
US       6043
India    3957
Name: count, dtype: int64

S2 country distribution:
country
US       3016817
India    2017799
Name: count, dtype: int64

S3 country distribution:
country
US       3170056
India    2115547
Name: count, dtype: int64

--- resource snapshot: after data load ---
  process RSS      : 3,106.7 MB
  system used/total: 3.74 / 31.35 GB
  disk used/total  : 0.00 / 19.52 GB
  disk free        : 19.50 GB
  elapsed since start: 62.4s

Total loading time: 3

## 6. Normalization
Same normalization rules as the reference `normalize.py` (lowercasing, accent stripping, punctuation removal, legal-suffix trimming for names, street-abbreviation folding for addresses). `country` is treated purely as an open-set string label — it is never mapped to a fixed vocabulary, matching the challenge's warning that TEST includes France, which never appears in TRAIN.

In [5]:

_LEGAL_SUFFIXES = [
    "incorporated", "inc", "corporation", "corp", "limited", "ltd", "llc",
    "company", "co", "private limited", "pvt ltd",
]

_ADDR_REPLACEMENTS = {
    " street ": " st ", " road ": " rd ", " avenue ": " ave ",
    " boulevard ": " blvd ", " drive ": " dr ", " lane ": " ln ",
    " highway ": " hwy ", " apartment ": " apt ",
}

_GENERIC_ADDRESS_WORDS = {
    "street", "st", "road", "rd", "avenue", "ave", "lane", "ln", "drive", "dr",
    "boulevard", "blvd", "highway", "hwy", "near", "opposite", "opp", "building",
    "bldg", "floor", "fl", "block", "district", "city", "state", "india",
    "private", "limited", "ltd", "pvt",
}


def normalize_text(text):
    if not text:
        return ""
    text = str(text).lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_business_name(name):
    name = normalize_text(name)
    words = name.split()
    while words and " ".join(words[-2:]) in _LEGAL_SUFFIXES:
        words = words[:-2]
    while words and words[-1] in _LEGAL_SUFFIXES:
        words = words[:-1]
    return " ".join(words)


def normalize_address(address):
    a = normalize_text(address)
    a = " " + a + " "
    for old, new in _ADDR_REPLACEMENTS.items():
        a = a.replace(old, new)
    return a.strip()


# Quick self-test against the reference module's own examples.
for name in ["Payne Enterprises", "Payne \u00c9nterprises", "Payne Enterprises LLC", "PAYNE-ENRTPRMISES"]:
    print(f"{name!r:30s} -> {normalize_business_name(name)!r}")
for addr in ["3315 Fremont Street, Peoria, IL", "3315 FREMONT ST, PEORIA, IL"]:
    print(f"{addr!r:35s} -> {normalize_address(addr)!r}")


'Payne Enterprises'            -> 'payne enterprises'
'Payne Énterprises'            -> 'payne enterprises'
'Payne Enterprises LLC'        -> 'payne enterprises'
'PAYNE-ENRTPRMISES'            -> 'payne enrtprmises'
'3315 Fremont Street, Peoria, IL'   -> '3315 fremont st peoria il'
'3315 FREMONT ST, PEORIA, IL'       -> '3315 fremont st peoria il'


## 7. Blocking strategies

Every blocking key from the reference's `blocking.py` is reproduced verbatim, since the reference
implementation's own measured recall (98.26% / 97.68%) already came from this exact key set:

| Key | Signal |
|---|---|
| `NP6` / `NP8` | normalized-name prefix (6 / 8 chars) |
| `NS6` | normalized-name suffix (6 chars) |
| `FL` / `FL3` | first-word + last-word combination |
| `F2` | first two words |
| `FML` | first + middle + last word |
| `UNP4` / `UNP6` / `UFL` | Unicode-preserving (non-ASCII-safe) versions of the above, for transliterated / non-Latin names |
| `AN` | address number + a nearby non-numeric token |
| `A2` | address number + first two non-numeric tokens |
| `AW2` / `AL2` | first-two / last-two non-numeric address tokens |
| `NL` | address number + last non-numeric token |
| `A4W` | address number + a nearby pair of informative tokens (captures reordered/landmark addresses) |
| `AT2` | a pair of the longest, most informative address tokens, position-independent |
| `RAT20` | any single rare, informative address token (frequency-capped at 20 occurrences) — recovers matches when the name is corrupted/transliterated but a distinctive address token survives |

All keys are composed with the record's (uppercased, stripped) `country` so blocking never crosses
country boundaries — this is safe because `country` is read as a free string, not a fixed set, so an
unseen TEST country (France) still blocks correctly against itself.


In [6]:

def make_name_keys(name):
    n = normalize_business_name(name)
    if not n:
        return []
    keys = []
    compact = n.replace(" ", "")
    if len(compact) >= 6:
        keys.append(("NP6", compact[:6]))
    if len(compact) >= 8:
        keys.append(("NP8", compact[:8]))
    if len(compact) >= 6:
        keys.append(("NS6", compact[-6:]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FL", first[:4] + "|" + last[:4]))
            keys.append(("FL3", first[:3] + "|" + last[:5]))
        w1, w2 = words[0], words[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("F2", w1[:5] + "|" + w2[:5]))
    if len(words) >= 3:
        first, middle, last = words[0], words[-2], words[-1]
        if len(first) >= 3 and len(last) >= 3:
            keys.append(("FML", first[:3] + "|" + middle[:3] + "|" + last[:3]))
    return keys


def make_unicode_name_keys(name):
    if not name:
        return []
    n = unicodedata.normalize("NFKC", str(name)).casefold()
    n = " ".join(n.split())
    if not n:
        return []
    keys = []
    compact = "".join(ch for ch in n if not ch.isspace())
    if len(compact) >= 4:
        keys.append(("UNP4", compact[:4]))
    if len(compact) >= 6:
        keys.append(("UNP6", compact[:6]))
    words = n.split()
    if len(words) >= 2:
        first, last = words[0], words[-1]
        if len(first) >= 2 and len(last) >= 2:
            keys.append(("UFL", first[:3] + "|" + last[:3]))
    return keys


def make_address_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    keys = []
    words = a.split()
    if not words:
        return []
    numbers = [w for w in words if w.isdigit()]
    non_numeric = [w for w in words if not w.isdigit()]
    if numbers:
        number = numbers[0]
        for word in words:
            if not word.isdigit() and len(word) >= 3:
                keys.append(("AN", number + "|" + word[:6]))
                break
        if len(non_numeric) >= 2:
            keys.append(("A2", number + "|" + non_numeric[0][:4] + "|" + non_numeric[1][:4]))
    if len(non_numeric) >= 2:
        w1, w2 = non_numeric[0], non_numeric[1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AW2", w1[:5] + "|" + w2[:5]))
        w1, w2 = non_numeric[-2], non_numeric[-1]
        if len(w1) >= 3 and len(w2) >= 3:
            keys.append(("AL2", w1[:5] + "|" + w2[:5]))
    if numbers and non_numeric:
        number = numbers[0]
        last_word = non_numeric[-1]
        if len(last_word) >= 3:
            keys.append(("NL", number + "|" + last_word[:6]))
    return keys


def make_rare_address_token_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    keys, seen = [], set()
    for tok in tokens:
        if tok.isdigit() or len(tok) < 4 or tok in _GENERIC_ADDRESS_WORDS:
            continue
        token = tok[:12]
        if token in seen:
            continue
        seen.add(token)
        keys.append(("RAT20", token))
    return keys


_A4_WINDOW = 3


def make_a4_keys(address, window=_A4_WINDOW):
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    number_positions = [i for i, t in enumerate(tokens) if t.isdigit()]
    if not number_positions:
        return []
    keys, seen = [], set()
    for pos in number_positions:
        number = tokens[pos]
        nearby = []
        for offset in range(1, window + 1):
            for idx in (pos - offset, pos + offset):
                if 0 <= idx < len(tokens):
                    tok = tokens[idx]
                    if not tok.isdigit() and len(tok) >= 2:
                        nearby.append(tok)
        nearby = list(dict.fromkeys(nearby))[:3]
        for i in range(len(nearby)):
            for j in range(i + 1, len(nearby)):
                w1, w2 = sorted((nearby[i][:6], nearby[j][:6]))
                key_value = number + "|" + w1 + "|" + w2
                if key_value not in seen:
                    seen.add(key_value)
                    keys.append(("A4W", key_value))
    return keys


def make_at2_keys(address):
    a = normalize_address(address)
    if not a:
        return []
    tokens = a.split()
    informative = [t for t in tokens if not t.isdigit() and len(t) >= 4 and t not in _GENERIC_ADDRESS_WORDS]
    if len(informative) < 2:
        return []
    informative = sorted(informative, key=lambda x: (-len(x), x))[:6]
    keys, seen = [], set()
    for i in range(len(informative) - 1):
        w1, w2 = informative[i][:8], informative[i + 1][:8]
        if w1 == w2:
            continue
        a1, a2 = sorted((w1, w2))
        key_value = a1 + "|" + a2
        if key_value not in seen:
            seen.add(key_value)
            keys.append(("AT2", key_value))
    return keys


NAME_KEY_FUNCS = [make_name_keys, make_unicode_name_keys]
ADDRESS_KEY_FUNCS = [make_address_keys, make_rare_address_token_keys, make_a4_keys, make_at2_keys]


def make_all_keys(name, address):
    keys = []
    for f in NAME_KEY_FUNCS:
        keys.extend(f(name))
    for f in ADDRESS_KEY_FUNCS:
        keys.extend(f(address))
    return keys


ALL_KEY_TYPES = sorted(DEFAULT_FREQ_CAPS.keys())
print("Blocking key types:", ALL_KEY_TYPES)


Blocking key types: ['A2', 'A4W', 'AL2', 'AN', 'AT2', 'AW2', 'F2', 'FL', 'FL3', 'FML', 'NL', 'NP6', 'NP8', 'NS6', 'RAT20', 'UFL', 'UNP4', 'UNP6']


## 8. Index construction

Builds a **country-aware inverted index** for S2 and S3: every (key_type, key_value) -> list of
matching entity suffixes, exactly like the reference `indexer.py` / `MemoryIndex`, but constructed
with a single vectorized `groupby` instead of a per-record Python `defaultdict` insert loop, and
looked up later with a `merge` instead of a per-S1-row dictionary walk. This is the main performance
fix over the reference implementation, which spent nearly all of its ~26 minutes on this exact step.

The frequency cap (`DEFAULT_FREQ_CAPS`) is applied once here, when the index is built, by dropping
any `(key_type, key_value)` bucket whose size exceeds that key type's cap — identical semantics to the
reference's per-lookup frequency check, just computed once instead of on every S1 row.


In [11]:
# ============================================================
# 8. INDEX CONSTRUCTION — KAGGLE 10K VERSION
# ============================================================

import gc
import time


def compose_key(country, key_value):
    return (
        (country or "").strip().upper()
        + "||"
        + key_value
    )


def build_key_table(df, dataset_name="DATASET"):
    """
    Build the WIP blocking-key table.

    IMPORTANT:
    This preserves make_all_keys() exactly.
    """

    t0 = time.time()

    recs = []

    ids = df["entity_id"].to_numpy()
    names = df["business_name"].to_numpy()
    addrs = df["business_address"].to_numpy()
    countries = df["country"].to_numpy()

    total = len(df)

    print()
    print(
        f"Building blocking keys for {dataset_name}: "
        f"{total:,} rows"
    )

    for row_num, (
        eid,
        name,
        addr,
        country
    ) in enumerate(
        zip(
            ids,
            names,
            addrs,
            countries
        ),
        start=1
    ):

        suffix = (
            eid.split("-", 1)[1]
            if "-" in eid
            else eid
        )

        country_prefix = (
            (country or "").strip().upper()
        )

        for kt, kv in make_all_keys(
            name,
            addr
        ):

            recs.append(
                (
                    kt,
                    country_prefix
                    + "||"
                    + kv,
                    suffix
                )
            )

        if (
            row_num == 1
            or row_num % 10_000 == 0
            or row_num == total
        ):

            elapsed = time.time() - t0

            print(
                f"{dataset_name}: "
                f"{row_num:,}/{total:,} rows | "
                f"raw keys={len(recs):,} | "
                f"elapsed={elapsed:.1f}s"
            )

    result = pd.DataFrame(
        recs,
        columns=[
            "key_type",
            "key_value",
            "suffix"
        ]
    )

    del recs
    gc.collect()

    print(
        f"{dataset_name}: generated "
        f"{len(result):,} raw keys "
        f"in {time.time()-t0:.1f}s"
    )

    return result


def build_filtered_index(
    key_table,
    caps=DEFAULT_FREQ_CAPS,
    default_cap=DEFAULT_FREQ_CAP_FALLBACK
):

    print(
        f"Filtering {len(key_table):,} raw keys..."
    )

    t0 = time.time()

    # --------------------------------------------------------
    # Bucket frequencies
    # --------------------------------------------------------

    sizes = (
        key_table
        .groupby(
            [
                "key_type",
                "key_value"
            ],
            sort=False
        )
        .size()
        .rename("bucket_size")
        .reset_index()
    )

    sizes["cap"] = (
        sizes["key_type"]
        .map(caps)
        .fillna(default_cap)
    )

    keep = sizes.loc[
        sizes["bucket_size"] <= sizes["cap"],
        [
            "key_type",
            "key_value",
            "bucket_size"
        ]
    ]

    print(
        f"Unique buckets : {len(sizes):,}"
    )

    print(
        f"Buckets kept   : {len(keep):,}"
    )

    # --------------------------------------------------------
    # Keep only valid buckets
    # --------------------------------------------------------

    filtered = key_table.merge(
        keep,
        on=[
            "key_type",
            "key_value"
        ],
        how="inner",
        sort=False
    )

    print(
        f"Filtered keys  : {len(filtered):,}"
    )

    print(
        f"Filtering time : {time.time()-t0:.1f}s"
    )

    return filtered, sizes


# ============================================================
# S2 INDEX
# ============================================================

t_idx = time.time()

print()
print("=" * 60)
print("BUILDING S2 INDEX")
print("=" * 60)

s2_key_table = build_key_table(
    S2,
    "S2"
)

s2_index, s2_bucket_sizes = build_filtered_index(
    s2_key_table
)

print()
print(
    f"S2 raw keys       : "
    f"{len(s2_key_table):,}"
)

print(
    f"S2 filtered keys  : "
    f"{len(s2_index):,}"
)

print(
    f"S2 unique buckets : "
    f"{len(s2_bucket_sizes):,}"
)

resource_snapshot(
    "after S2 index"
)


# ============================================================
# S3 INDEX
# ============================================================

print()
print("=" * 60)
print("BUILDING S3 INDEX")
print("=" * 60)

s3_key_table = build_key_table(
    S3,
    "S3"
)

s3_index, s3_bucket_sizes = build_filtered_index(
    s3_key_table
)

print()
print(
    f"S3 raw keys       : "
    f"{len(s3_key_table):,}"
)

print(
    f"S3 filtered keys  : "
    f"{len(s3_index):,}"
)

print(
    f"S3 unique buckets : "
    f"{len(s3_bucket_sizes):,}"
)

resource_snapshot(
    "after S3 index"
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 60)
print("INDEX CONSTRUCTION COMPLETE")
print("=" * 60)

print(
    f"S2 filtered keys : {len(s2_index):,}"
)

print(
    f"S3 filtered keys : {len(s3_index):,}"
)

print(
    f"Total time       : "
    f"{time.time()-t_idx:.1f}s"
)

resource_snapshot(
    "after index construction"
)


BUILDING S2 INDEX

Building blocking keys for S2: 100,000 rows
S2: 1/100,000 rows | raw keys=16 | elapsed=0.0s
S2: 10,000/100,000 rows | raw keys=206,013 | elapsed=1.2s
S2: 20,000/100,000 rows | raw keys=412,522 | elapsed=2.3s
S2: 30,000/100,000 rows | raw keys=617,988 | elapsed=3.5s
S2: 40,000/100,000 rows | raw keys=823,665 | elapsed=4.6s
S2: 50,000/100,000 rows | raw keys=1,029,652 | elapsed=5.7s
S2: 60,000/100,000 rows | raw keys=1,236,229 | elapsed=6.9s
S2: 70,000/100,000 rows | raw keys=1,443,099 | elapsed=8.0s
S2: 80,000/100,000 rows | raw keys=1,650,424 | elapsed=9.2s
S2: 90,000/100,000 rows | raw keys=1,857,192 | elapsed=10.3s
S2: 100,000/100,000 rows | raw keys=2,063,701 | elapsed=11.5s
S2: generated 2,063,701 raw keys in 12.2s
Filtering 2,063,701 raw keys...
Unique buckets : 1,198,214
Buckets kept   : 1,195,103
Filtered keys  : 1,732,721
Filtering time : 3.6s

S2 raw keys       : 2,063,701
S2 filtered keys  : 1,732,721
S2 unique buckets : 1,198,214
--- resource snapshot: af

'--- resource snapshot: after index construction ---\n  process RSS      : 1,207.4 MB\n  system used/total: 1.94 / 31.35 GB\n  disk used/total  : 0.00 / 19.52 GB\n  disk free        : 19.50 GB\n  elapsed since start: 410.3s'

In [12]:
def build_s1_key_table(s1_chunk):
    recs = []
    ids = s1_chunk["entity_id"].to_numpy()
    names = s1_chunk["business_name"].to_numpy()
    addrs = s1_chunk["business_address"].to_numpy()
    countries = s1_chunk["country"].to_numpy()

    for eid, name, addr, country in zip(
        ids, names, addrs, countries
    ):
        c = compose_key(country, "")[:-2]

        for kt, kv in make_all_keys(name, addr):
            recs.append(
                (eid, kt, c + "||" + kv)
            )

    return pd.DataFrame(
        recs,
        columns=[
            "s1_id",
            "key_type",
            "key_value"
        ]
    )


def generate_candidates(
    s1_df,
    index_df,
    source_prefix,
    chunk_size=S1_CHUNK_SIZE
):

    out_chunks = []

    n = len(s1_df)

    for start in range(
        0,
        n,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            n
        )

        chunk = s1_df.iloc[
            start:end
        ]

        print(
            f"{source_prefix}: "
            f"S1 {start:,}-{end:,}"
        )

        s1_keys = build_s1_key_table(
            chunk
        )

        merged = s1_keys.merge(
            index_df,
            on=[
                "key_type",
                "key_value"
            ],
            how="inner"
        )

        if merged.empty:
            continue

        merged["candidate_id"] = (
            source_prefix
            + "-"
            + merged["suffix"]
        )

        agg = (
            merged
            .groupby(
                [
                    "s1_id",
                    "candidate_id"
                ]
            )
            .agg(
                key_types=(
                    "key_type",
                    lambda s:
                    "|".join(
                        sorted(
                            set(s)
                        )
                    )
                ),
                num_keys=(
                    "key_type",
                    "nunique"
                ),
                min_freq=(
                    "bucket_size",
                    "min"
                )
            )
            .reset_index()
        )

        out_chunks.append(
            agg
        )

        print(
            f"  candidates in chunk: "
            f"{len(agg):,}"
        )

        del s1_keys
        del merged
        del agg
        gc.collect()

    if not out_chunks:

        return pd.DataFrame(
            columns=[
                "s1_id",
                "candidate_id",
                "key_types",
                "num_keys",
                "min_freq"
            ]
        )

    result = pd.concat(
        out_chunks,
        ignore_index=True
    )

    del out_chunks
    gc.collect()

    return result


# ============================================================
# RUN CANDIDATE GENERATION
# ============================================================

t_cand = time.time()

s2_candidates = generate_candidates(
    S1,
    s2_index,
    "S2"
)

print(
    f"S2 candidate pairs: "
    f"{len(s2_candidates):,}"
)

resource_snapshot(
    "after S2 candidates"
)


s3_candidates = generate_candidates(
    S1,
    s3_index,
    "S3"
)

print(
    f"S3 candidate pairs: "
    f"{len(s3_candidates):,}"
)

resource_snapshot(
    "after S3 candidates"
)


print()
print("=" * 60)
print("CANDIDATE GENERATION COMPLETE")
print("=" * 60)

print(
    f"S2 candidate pairs : "
    f"{len(s2_candidates):,}"
)

print(
    f"S3 candidate pairs : "
    f"{len(s3_candidates):,}"
)

print(
    f"Combined pairs     : "
    f"{len(s2_candidates) + len(s3_candidates):,}"
)

print(
    f"Total time         : "
    f"{time.time() - t_cand:.1f}s"
)

resource_snapshot(
    "after candidate generation"
)

S2: S1 0-2,500
  candidates in chunk: 305,487
S2: S1 2,500-5,000
  candidates in chunk: 319,982
S2: S1 5,000-7,500
  candidates in chunk: 311,383
S2: S1 7,500-10,000
  candidates in chunk: 322,205
S2 candidate pairs: 1,259,057
--- resource snapshot: after S2 candidates ---
  process RSS      : 1,266.9 MB
  system used/total: 2.07 / 31.35 GB
  disk used/total  : 0.00 / 19.52 GB
  disk free        : 19.50 GB
  elapsed since start: 708.3s
S3: S1 0-2,500
  candidates in chunk: 315,160
S3: S1 2,500-5,000
  candidates in chunk: 333,725
S3: S1 5,000-7,500
  candidates in chunk: 324,806
S3: S1 7,500-10,000
  candidates in chunk: 334,090
S3 candidate pairs: 1,307,781
--- resource snapshot: after S3 candidates ---
  process RSS      : 1,264.7 MB
  system used/total: 1.92 / 31.35 GB
  disk used/total  : 0.00 / 19.52 GB
  disk free        : 19.50 GB
  elapsed since start: 737.5s

CANDIDATE GENERATION COMPLETE
S2 candidate pairs : 1,259,057
S3 candidate pairs : 1,307,781
Combined pairs     : 2,566,

'--- resource snapshot: after candidate generation ---\n  process RSS      : 1,264.7 MB\n  system used/total: 1.92 / 31.35 GB\n  disk used/total  : 0.00 / 19.52 GB\n  disk free        : 19.50 GB\n  elapsed since start: 737.5s'

In [13]:
# ============================================================
# CANDIDATE RECALL EVALUATION
# ============================================================

import ast
import pandas as pd

print("=" * 60)
print("CANDIDATE RECALL EVALUATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Prepare ground truth for the 10K S1 development set
# ------------------------------------------------------------

gt = GT_RAW.copy()

print("GT rows available:", len(gt))
print("GT columns:", list(gt.columns))


# Keep only the S1 IDs currently in our development set
s1_ids = set(S1["entity_id"].astype(str))

gt["source1_entity_id"] = (
    gt["source1_entity_id"].astype(str)
)

gt = gt[
    gt["source1_entity_id"].isin(s1_ids)
].copy()


# ------------------------------------------------------------
# 2. Parse matched_entity_ids
# ------------------------------------------------------------

def parse_matches(x):

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return [str(v) for v in x]

    x = str(x).strip()

    if not x:
        return []

    # Try Python-list format:
    # "['S2-123', 'S3-456']"
    try:
        parsed = ast.literal_eval(x)

        if isinstance(parsed, list):
            return [str(v) for v in parsed]

    except Exception:
        pass

    # Fallback: remove common brackets/quotes
    x = x.strip("[]")

    if not x:
        return []

    parts = x.split(",")

    return [
        p.strip().strip("'").strip('"')
        for p in parts
        if p.strip()
    ]


gt["true_matches"] = gt[
    "matched_entity_ids"
].apply(parse_matches)


# ------------------------------------------------------------
# 3. Convert candidate results into lookup sets
# ------------------------------------------------------------

s2_candidate_lookup = (
    s2_candidates
    .groupby("s1_id")["candidate_id"]
    .agg(lambda x: set(x.astype(str)))
    .to_dict()
)

s3_candidate_lookup = (
    s3_candidates
    .groupby("s1_id")["candidate_id"]
    .agg(lambda x: set(x.astype(str)))
    .to_dict()
)


# ------------------------------------------------------------
# 4. Calculate recall
# ------------------------------------------------------------

s2_true = 0
s2_found = 0

s3_true = 0
s3_found = 0

combined_true = 0
combined_found = 0

rows_with_gt = 0


for _, row in gt.iterrows():

    s1_id = str(
        row["source1_entity_id"]
    )

    true_matches = set(
        row["true_matches"]
    )

    if not true_matches:
        continue

    rows_with_gt += 1

    s2_true_matches = {
        x for x in true_matches
        if x.startswith("S2-")
    }

    s3_true_matches = {
        x for x in true_matches
        if x.startswith("S3-")
    }

    s2_candidates_set = (
        s2_candidate_lookup.get(
            s1_id,
            set()
        )
    )

    s3_candidates_set = (
        s3_candidate_lookup.get(
            s1_id,
            set()
        )
    )

    s2_true += len(
        s2_true_matches
    )

    s2_found += len(
        s2_true_matches
        & s2_candidates_set
    )

    s3_true += len(
        s3_true_matches
    )

    s3_found += len(
        s3_true_matches
        & s3_candidates_set
    )

    combined_true += len(
        true_matches
    )

    combined_found += len(
        true_matches
        & (
            s2_candidates_set
            | s3_candidates_set
        )
    )


# ------------------------------------------------------------
# 5. Print results
# ------------------------------------------------------------

s2_recall = (
    s2_found / s2_true
    if s2_true else 1.0
)

s3_recall = (
    s3_found / s3_true
    if s3_true else 1.0
)

combined_recall = (
    combined_found / combined_true
    if combined_true else 1.0
)


print()
print("=" * 60)
print("GROUND-TRUTH MATCH COVERAGE")
print("=" * 60)

print(f"GT S1 rows with matches : {rows_with_gt:,}")

print()
print(
    f"S2 true matches found   : "
    f"{s2_found:,} / {s2_true:,}"
)

print(
    f"S2 candidate recall     : "
    f"{s2_recall:.4%}"
)

print()
print(
    f"S3 true matches found   : "
    f"{s3_found:,} / {s3_true:,}"
)

print(
    f"S3 candidate recall     : "
    f"{s3_recall:.4%}"
)

print()
print(
    f"Combined matches found  : "
    f"{combined_found:,} / {combined_true:,}"
)

print(
    f"Combined candidate recall: "
    f"{combined_recall:.4%}"
)

print()
print("=" * 60)
print("CANDIDATE RECALL COMPLETE")
print("=" * 60)

CANDIDATE RECALL EVALUATION
GT rows available: 10000
GT columns: ['source1_entity_id', 'matched_entity_ids']

GROUND-TRUTH MATCH COVERAGE
GT S1 rows with matches : 9,440

S2 true matches found   : 314 / 16,765
S2 candidate recall     : 1.8729%

S3 true matches found   : 329 / 17,987
S3 candidate recall     : 1.8291%

Combined matches found  : 643 / 34,752
Combined candidate recall: 1.8503%

CANDIDATE RECALL COMPLETE


In [14]:
# ============================================================
# DEBUG: CHECK GT ID FORMAT VS CANDIDATE ID FORMAT
# ============================================================

print("=" * 60)
print("ID FORMAT DEBUG")
print("=" * 60)

print("\n--- GT examples ---")
print(
    GT_RAW[
        ["source1_entity_id", "matched_entity_ids"]
    ].head(10).to_string(index=False)
)

print("\n--- S1 examples ---")
print(
    S1["entity_id"].head(10).tolist()
)

print("\n--- S2 entity_id examples ---")
print(
    S2["entity_id"].head(10).tolist()
)

print("\n--- S3 entity_id examples ---")
print(
    S3["entity_id"].head(10).tolist()
)

print("\n--- Candidate ID examples ---")

print(
    "S2 candidate IDs:",
    s2_candidates["candidate_id"].head(20).tolist()
)

print(
    "S3 candidate IDs:",
    s3_candidates["candidate_id"].head(20).tolist()
)


# ------------------------------------------------------------
# Check actual suffix construction
# ------------------------------------------------------------

print("\n--- Index suffix examples ---")

print(
    s2_index[
        ["suffix", "key_type", "key_value"]
    ].head(10).to_string(index=False)
)

print(
    s3_index[
        ["suffix", "key_type", "key_value"]
    ].head(10).to_string(index=False)
)


# ------------------------------------------------------------
# Compare GT IDs with candidate IDs
# ------------------------------------------------------------

gt_match_ids = set()

for x in GT_RAW["matched_entity_ids"].dropna().head(1000):

    parsed = parse_matches(x)

    for pid in parsed:
        gt_match_ids.add(str(pid))


s2_cand_ids = set(
    s2_candidates["candidate_id"]
    .astype(str)
    .head(100000)
)

s3_cand_ids = set(
    s3_candidates["candidate_id"]
    .astype(str)
    .head(100000)
)

print("\n--- Format comparison ---")

print(
    "GT match examples:",
    list(gt_match_ids)[:20]
)

print(
    "Candidate S2 examples:",
    list(s2_cand_ids)[:20]
)

print(
    "Candidate S3 examples:",
    list(s3_cand_ids)[:20]
)

print("\nGT IDs beginning with S2-:",
      sum(x.startswith("S2-") for x in gt_match_ids))

print("GT IDs beginning with S3-:",
      sum(x.startswith("S3-") for x in gt_match_ids))

print("\nCandidate S2 IDs beginning with S2-:",
      sum(x.startswith("S2-") for x in s2_cand_ids))

print("Candidate S3 IDs beginning with S3-:",
      sum(x.startswith("S3-") for x in s3_cand_ids))

print("=" * 60)

ID FORMAT DEBUG

--- GT examples ---
source1_entity_id                                                                         matched_entity_ids
     S1-424976769                                                      S2-18126063,S3-443109567,S3-296472077
     S1-874660306                                                                  S2-279857860,S3-289365741
     S1-539879230                                         S2-83796020,S3-945547250,S3-127246506,S3-117651018
     S1-840168502                                                                                           
     S1-716830359                                        S2-795817041,S2-306163710,S3-762649005,S3-737589534
     S1-610149670                           S2-601177655,S2-809918482,S3-588852284,S3-339503285,S3-744911212
     S1-210849781 S2-562117280,S2-845708664,S2-820221567,S3-649907207,S3-895784814,S3-210870050,S3-240268161
     S1-844896591                                                                   S2-3622

In [15]:
# ============================================================
# CORRECT CANDIDATE RECALL
# ONLY EVALUATE GT MATCHES PRESENT IN DEV S2/S3
# ============================================================

print("=" * 60)
print("CORRECTED CANDIDATE RECALL")
print("=" * 60)


# ------------------------------------------------------------
# IDs actually available in our 100K development datasets
# ------------------------------------------------------------

dev_s2_ids = set(
    S2["entity_id"].astype(str)
)

dev_s3_ids = set(
    S3["entity_id"].astype(str)
)

print(f"Dev S2 IDs: {len(dev_s2_ids):,}")
print(f"Dev S3 IDs: {len(dev_s3_ids):,}")


# ------------------------------------------------------------
# Candidate lookup
# ------------------------------------------------------------

s2_candidate_lookup = (
    s2_candidates
    .groupby("s1_id")["candidate_id"]
    .agg(lambda x: set(x.astype(str)))
    .to_dict()
)

s3_candidate_lookup = (
    s3_candidates
    .groupby("s1_id")["candidate_id"]
    .agg(lambda x: set(x.astype(str)))
    .to_dict()
)


# ------------------------------------------------------------
# Evaluate ONLY matches existing in our dev S2/S3
# ------------------------------------------------------------

s2_true = 0
s2_found = 0

s3_true = 0
s3_found = 0

combined_true = 0
combined_found = 0

s1_with_eval_matches = 0


for _, row in gt.iterrows():

    s1_id = str(row["source1_entity_id"])

    true_matches = set(row["true_matches"])

    # Only GT matches that actually exist in our
    # 100K development S2/S3 datasets
    eval_s2 = {
        x for x in true_matches
        if x in dev_s2_ids
    }

    eval_s3 = {
        x for x in true_matches
        if x in dev_s3_ids
    }

    eval_combined = eval_s2 | eval_s3

    if eval_combined:
        s1_with_eval_matches += 1

    s2_true += len(eval_s2)
    s3_true += len(eval_s3)
    combined_true += len(eval_combined)

    s2_found += len(
        eval_s2 &
        s2_candidate_lookup.get(
            s1_id,
            set()
        )
    )

    s3_found += len(
        eval_s3 &
        s3_candidate_lookup.get(
            s1_id,
            set()
        )
    )

    combined_found += len(
        eval_combined &
        (
            s2_candidate_lookup.get(
                s1_id,
                set()
            )
            |
            s3_candidate_lookup.get(
                s1_id,
                set()
            )
        )
    )


# ------------------------------------------------------------
# Recall
# ------------------------------------------------------------

s2_recall = (
    s2_found / s2_true
    if s2_true else 0
)

s3_recall = (
    s3_found / s3_true
    if s3_true else 0
)

combined_recall = (
    combined_found / combined_true
    if combined_true else 0
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("=" * 60)
print("DEVELOPMENT-DATA RECALL")
print("=" * 60)

print(
    f"S1 rows with at least one "
    f"dev-set GT match : {s1_with_eval_matches:,}"
)

print()

print(
    f"S2 matches in dev set : "
    f"{s2_true:,}"
)

print(
    f"S2 found by blockers  : "
    f"{s2_found:,}"
)

print(
    f"S2 candidate recall   : "
    f"{s2_recall:.4%}"
)

print()

print(
    f"S3 matches in dev set : "
    f"{s3_true:,}"
)

print(
    f"S3 found by blockers  : "
    f"{s3_found:,}"
)

print(
    f"S3 candidate recall   : "
    f"{s3_recall:.4%}"
)

print()

print(
    f"Combined matches in dev set : "
    f"{combined_true:,}"
)

print(
    f"Combined found              : "
    f"{combined_found:,}"
)

print(
    f"Combined candidate recall   : "
    f"{combined_recall:.4%}"
)

print()
print("=" * 60)
print("CORRECTED RECALL COMPLETE")
print("=" * 60)

CORRECTED CANDIDATE RECALL
Dev S2 IDs: 100,000
Dev S3 IDs: 100,000

DEVELOPMENT-DATA RECALL
S1 rows with at least one dev-set GT match : 628

S2 matches in dev set : 316
S2 found by blockers  : 314
S2 candidate recall   : 99.3671%

S3 matches in dev set : 334
S3 found by blockers  : 329
S3 candidate recall   : 98.5030%

Combined matches in dev set : 650
Combined found              : 643
Combined candidate recall   : 98.9231%

CORRECTED RECALL COMPLETE


## 10K Development Candidate Recall

Candidate generation was evaluated on the 10,000 S1 development records against the S2 and S3 records available in the 100,000-row development subsets.

| Metric | Result |
|---|---:|
| Development S1 records | 10,000 |
| Development S2 records | 100,000 |
| Development S3 records | 100,000 |
| S2 candidate recall | 99.37% |
| S3 candidate recall | 98.50% |
| Combined candidate recall | 98.92% |
| Combined candidate pairs | 2,566,838 |

The blocking stage successfully retains the vast majority of ground-truth matches while reducing the search space to a manageable candidate set. The combined candidate recall of **98.92%** indicates that the blocking strategy is suitable for the subsequent candidate-ranking and matching stages.

## 9. Candidate generation

S1 is processed in chunks of `S1_CHUNK_SIZE` rows (a no-op at 10K, but this is what keeps the
full-scale run from ever materializing a single all-pairs candidate table). For each chunk, S1's
blocking keys are generated, merged against the pre-built S2/S3 index, and deduplicated per
`(s1_id, candidate_id)` — recording which key types hit and the smallest bucket size among them
(useful later for weak-vs-strong-candidate features).


In [9]:
print("s2_index exists:", "s2_index" in globals())
print("s3_index exists:", "s3_index" in globals())

# Show similar variables that actually exist
print("\nIndex-like variables:")
print([
    x for x in globals()
    if "index" in x.lower()
])

s2_index exists: False
s3_index exists: False

Index-like variables:
['generate_candidates_from_streaming_index']


In [10]:
print("build_index exists:", "build_index" in globals())
print("MemoryIndex exists:", "MemoryIndex" in globals())
print("SqliteIndex exists:", "SqliteIndex" in globals())

print("\nCurrent important variables:")
for name in [
    "S1", "S2", "S3",
    "GT_RAW",
    "GROUND_TRUTH",
    "build_index",
    "MemoryIndex",
    "SqliteIndex",
]:
    print(
        f"{name}:",
        type(globals()[name]).__name__
        if name in globals()
        else "NOT DEFINED"
    )

build_index exists: False
MemoryIndex exists: False
SqliteIndex exists: False

Current important variables:
S1: DataFrame
S2: DataFrame
S3: DataFrame
GT_RAW: DataFrame
GROUND_TRUTH: NOT DEFINED
build_index: NOT DEFINED
MemoryIndex: NOT DEFINED
SqliteIndex: NOT DEFINED


In [8]:
# ============================================================
# CANDIDATE GENERATION — STREAMING VERSION
# ============================================================
#
# IMPORTANT:
# Do NOT use the old generate_candidates() function here.
#
# The new blocking stage creates:
#     s2_index
#     s3_index
#
# as compact in-memory dictionaries.
#
# This function performs direct dictionary lookups and avoids
# pandas merge() over a large blocking-key table.
# ============================================================

import gc
import time


def generate_candidates_from_streaming_index(
    s1_df,
    index,
    source_prefix,
    chunk_size=2_500
):
    """
    Generate candidates from the compact streaming index.

    Output columns:
        s1_id
        candidate_id
        key_types
        num_keys
        min_freq
    """

    start_time = time.time()

    output_chunks = []

    total = len(s1_df)

    print()
    print("=" * 65)
    print(
        f"GENERATING {source_prefix} CANDIDATES"
    )
    print("=" * 65)

    print(
        f"S1 rows       : {total:,}"
    )

    print(
        f"Index buckets : {len(index):,}"
    )

    print(
        f"Chunk size    : {chunk_size:,}"
    )

    print()

    for start in range(
        0,
        total,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total
        )

        chunk = s1_df.iloc[
            start:end
        ]

        # ----------------------------------------------------
        # Candidate accumulator for this chunk only
        # ----------------------------------------------------

        candidate_map = {}

        rows = chunk[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country"
            ]
        ].itertuples(
            index=False,
            name=None
        )

        for eid, name, address, country in rows:

            eid = str(eid)

            country_key = (
                str(country).strip().upper()
                if country is not None
                else ""
            )

            per_record = candidate_map.setdefault(
                eid,
                {}
            )

            # IMPORTANT:
            # Use the exact same make_all_keys() function
            # from the WIP.
            keys = make_all_keys(
                name,
                address
            )

            for key_type, key_value in keys:

                lookup_key = (
                    key_type,
                    country_key
                    + "||"
                    + str(key_value)
                )

                bucket = index.get(
                    lookup_key
                )

                if not bucket:
                    continue

                bucket_size = len(bucket)

                for suffix in bucket:

                    candidate_id = (
                        source_prefix
                        + "-"
                        + str(suffix)
                    )

                    info = per_record.get(
                        candidate_id
                    )

                    if info is None:

                        per_record[candidate_id] = {
                            "key_types": {key_type},
                            "num_keys": 1,
                            "min_freq": bucket_size
                        }

                    else:

                        if key_type not in info["key_types"]:

                            info["key_types"].add(
                                key_type
                            )

                            info["num_keys"] += 1

                        if bucket_size < info["min_freq"]:

                            info["min_freq"] = bucket_size

        # ----------------------------------------------------
        # Convert THIS chunk only to DataFrame
        # ----------------------------------------------------

        rows_out = []

        for s1_id, candidates in candidate_map.items():

            for candidate_id, info in candidates.items():

                rows_out.append(
                    (
                        s1_id,
                        candidate_id,
                        "|".join(
                            sorted(
                                info["key_types"]
                            )
                        ),
                        info["num_keys"],
                        info["min_freq"]
                    )
                )

        if rows_out:

            result_chunk = pd.DataFrame(
                rows_out,
                columns=[
                    "s1_id",
                    "candidate_id",
                    "key_types",
                    "num_keys",
                    "min_freq"
                ]
            )

            output_chunks.append(
                result_chunk
            )

            candidate_count = len(
                result_chunk
            )

        else:

            candidate_count = 0

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        elapsed = time.time() - start_time

        print(
            f"{source_prefix}: "
            f"{end:,}/{total:,} S1 rows | "
            f"candidates={candidate_count:,} | "
            f"elapsed={elapsed:.1f}s"
        )

        del candidate_map
        del rows_out
        del chunk

        if "result_chunk" in locals():
            del result_chunk

        gc.collect()

    # --------------------------------------------------------
    # Combine output chunks
    # --------------------------------------------------------

    if not output_chunks:

        result = pd.DataFrame(
            columns=[
                "s1_id",
                "candidate_id",
                "key_types",
                "num_keys",
                "min_freq"
            ]
        )

    else:

        result = pd.concat(
            output_chunks,
            ignore_index=True
        )

        del output_chunks

        gc.collect()

        # Defensive deduplication
        result = result.drop_duplicates(
            subset=[
                "s1_id",
                "candidate_id"
            ],
            keep="first"
        ).reset_index(
            drop=True
        )

    elapsed = time.time() - start_time

    print()
    print(
        f"{source_prefix} candidate generation COMPLETE"
    )

    print(
        f"Candidate pairs: {len(result):,}"
    )

    print(
        f"Time: {elapsed:.1f}s"
    )

    return result


# ============================================================
# SAFETY CHECK
# ============================================================

if "s2_index" not in globals():
    raise RuntimeError(
        "s2_index does not exist. "
        "Run the STREAMING BLOCKING cell first."
    )

if "s3_index" not in globals():
    raise RuntimeError(
        "s3_index does not exist. "
        "Run the STREAMING BLOCKING cell first."
    )


# ============================================================
# S2 CANDIDATES
# ============================================================

t_cand = time.time()

s2_candidates = generate_candidates_from_streaming_index(
    S1,
    s2_index,
    "S2",
    chunk_size=2_500
)


# ============================================================
# S3 CANDIDATES
# ============================================================

s3_candidates = generate_candidates_from_streaming_index(
    S1,
    s3_index,
    "S3",
    chunk_size=2_500
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 65)
print("CANDIDATE GENERATION COMPLETE")
print("=" * 65)

print(
    f"S2 candidate pairs : "
    f"{len(s2_candidates):,}"
)

print(
    f"S3 candidate pairs : "
    f"{len(s3_candidates):,}"
)

print(
    f"Combined pairs     : "
    f"{len(s2_candidates) + len(s3_candidates):,}"
)

print(
    f"Total time         : "
    f"{time.time() - t_cand:.1f}s"
)

resource_snapshot(
    "after candidate generation"
)

RuntimeError: s2_index does not exist. Run the STREAMING BLOCKING cell first.

## 10. Candidate recall evaluation
Measured against `train_ground_truth.tsv` — nothing here is assumed or hard-coded.

In [ ]:

def parse_ground_truth(gt_df):
    """s1_id -> {'S2': set(...), 'S3': set(...), 'all': set(...)}"""
    out = {}
    for s1id, raw in zip(gt_df["source1_entity_id"], gt_df["matched_entity_ids"]):
        raw = (raw or "").strip()
        ids = {x.strip() for x in raw.split(",") if x.strip()} if raw else set()
        out[s1id] = {
            "S2": {x for x in ids if x.startswith("S2-")},
            "S3": {x for x in ids if x.startswith("S3-")},
            "all": ids,
        }
    return out


GROUND_TRUTH = parse_ground_truth(GT_RAW)
GT_PAIR_SET = {s1id + "|" + cid for s1id, entry in GROUND_TRUTH.items() for cid in entry["all"]}


def evaluate_candidate_recall(s1_ids, s2_cands, s3_cands, ground_truth, label="CANDIDATE RECALL"):
    s2_sets = s2_cands.groupby("s1_id")["candidate_id"].apply(set).to_dict()
    s3_sets = s3_cands.groupby("s1_id")["candidate_id"].apply(set).to_dict()

    s2_actual = s2_found = s3_actual = s3_found = comb_actual = comb_found = 0
    total_cand = 0
    for s1id in s1_ids:
        entry = ground_truth.get(s1id, {"S2": set(), "S3": set(), "all": set()})
        s2c = s2_sets.get(s1id, set())
        s3c = s3_sets.get(s1id, set())
        s2_actual += len(entry["S2"]); s2_found += len(entry["S2"] & s2c)
        s3_actual += len(entry["S3"]); s3_found += len(entry["S3"] & s3c)
        comb_actual += len(entry["all"]); comb_found += len(entry["all"] & (s2c | s3c))
        total_cand += len(s2c) + len(s3c)

    n = len(s1_ids)
    s2_recall = s2_found / s2_actual if s2_actual else 0.0
    s3_recall = s3_found / s3_actual if s3_actual else 0.0
    comb_recall = comb_found / comb_actual if comb_actual else 0.0

    report = {
        "n_s1": n,
        "s2_actual": s2_actual, "s2_found": s2_found, "s2_recall": s2_recall,
        "s3_actual": s3_actual, "s3_found": s3_found, "s3_recall": s3_recall,
        "comb_actual": comb_actual, "comb_found": comb_found, "comb_recall": comb_recall,
        "total_candidates": total_cand, "avg_candidates_per_s1": total_cand / n if n else 0.0,
    }

    print("=" * 60)
    print(label)
    print("=" * 60)
    print(f"S1 rows evaluated   : {n:,}")
    print(f"[S2] actual={s2_actual:,} found={s2_found:,} recall={s2_recall:.4f} ({s2_recall*100:.2f}%)")
    print(f"[S3] actual={s3_actual:,} found={s3_found:,} recall={s3_recall:.4f} ({s3_recall*100:.2f}%)")
    print(f"[COMBINED] actual={comb_actual:,} found={comb_found:,} recall={comb_recall:.4f} ({comb_recall*100:.2f}%)")
    print(f"Avg candidates/S1   : {report['avg_candidates_per_s1']:.2f}")
    print("=" * 60)
    return report


baseline_recall_report = evaluate_candidate_recall(
    S1["entity_id"].tolist(), s2_candidates, s3_candidates, GROUND_TRUTH,
    label="10K BASELINE — REPRODUCED CANDIDATE RECALL",
)

print()
print("Reference implementation reported (output/pipeline_report_train_10k.txt):")
print("  S2 recall: 98.26%  |  S3 recall: 97.68%  |  Combined: 97.96%  |  avg candidates/S1: 254.19")
print(f"This notebook measured:")
print(f"  S2 recall: {baseline_recall_report['s2_recall']*100:.2f}%  |  "
      f"S3 recall: {baseline_recall_report['s3_recall']*100:.2f}%  |  "
      f"Combined: {baseline_recall_report['comb_recall']*100:.2f}%  |  "
      f"avg candidates/S1: {baseline_recall_report['avg_candidates_per_s1']:.2f}")


### 10b. Per-blocker contribution analysis

Before trusting any blocker, measure it in isolation: for each key type, generate candidates using
*only* that key type (still frequency-capped) and report its standalone recall, candidates/S1 and
runtime. This is how redundant or low-value blockers are identified — by measurement, never by
guessing from the key's name.


In [ ]:
# Optional development diagnostic — deliberately OFF in the Kaggle-safe notebook.
# The WIP report already measured per-blocker contributions. Re-running this loop
# repeatedly merges every blocker and adds avoidable runtime.
if RUN_BLOCKER_DIAGNOSTICS:
    print("RUN_BLOCKER_DIAGNOSTICS=True: running optional blocker diagnostics...")
    blocker_rows = []
    s1_ids_list = S1["entity_id"].tolist()
    for kt in ALL_KEY_TYPES:
        t0 = time.time()
        c2 = candidates_for_single_key_type(S1, s2_index, "S2", kt)
        c3 = candidates_for_single_key_type(S1, s3_index, "S3", kt)
        dt = time.time() - t0
        s2_sets = c2.groupby("s1_id")["candidate_id"].apply(set).to_dict() if len(c2) else {}
        s3_sets = c3.groupby("s1_id")["candidate_id"].apply(set).to_dict() if len(c3) else {}
        found = 0
        total_cand = 0
        for s1id in s1_ids_list:
            entry = GROUND_TRUTH.get(s1id, {"all": set()})
            cset = s2_sets.get(s1id, set()) | s3_sets.get(s1id, set())
            found += len(entry["all"] & cset)
            total_cand += len(cset)
        recall = found / baseline_recall_report["comb_actual"] if baseline_recall_report["comb_actual"] else 0.0
        blocker_rows.append({
            "key_type": kt, "candidates": total_cand, "true_matches_found": found,
            "recall_contribution": recall, "candidates_per_s1": total_cand / len(s1_ids_list),
            "runtime_s": dt,
        })
        del c2, c3
        gc.collect()
    blocker_report = pd.DataFrame(blocker_rows).sort_values("recall_contribution", ascending=False)
    blocker_report.to_csv(REPORTS_DIR / "blocker_contribution_report.csv", index=False)
    print(blocker_report.to_string(index=False))
else:
    print("Skipped optional blocker contribution diagnostics.")


## 11. Feature engineering

Name / address / country features for every candidate pair. **RapidFuzz** (C++, vectorizable via
`process.cdist`-style batch calls) is used when available; otherwise a pure-Python character-trigram
and token Jaccard fallback is used, which is slower than RapidFuzz but still ~150K pairs/sec — the
notebook measured full-10K-scale feature computation at well under a minute even without RapidFuzz,
so it never silently falls back to the old, impractically slow `difflib.SequenceMatcher` per-pair loop.

Feature set: normalized exact match, character similarity, token-set Jaccard, prefix similarity,
address-number agreement, country agreement, and missing-field indicators — deliberately compact
rather than hundreds of expensive features, per the challenge's own resource-budget concerns.


In [16]:
print("=" * 60)
print("FEATURE ENGINEERING SAFETY CHECK")
print("=" * 60)

print("HAVE_RAPIDFUZZ:", HAVE_RAPIDFUZZ)

print("S2 candidate pairs:", f"{len(s2_candidates):,}")
print("S3 candidate pairs:", f"{len(s3_candidates):,}")
print(
    "Total candidate pairs:",
    f"{len(s2_candidates) + len(s3_candidates):,}"
)

print("GT_PAIR_SET exists:", "GT_PAIR_SET" in globals())

print()
print("Required variables:")

for name in [
    "S1",
    "S2",
    "S3",
    "s2_candidates",
    "s3_candidates",
    "normalize_business_name",
    "normalize_address",
    "make_all_keys",
]:
    print(
        f"{name}:",
        "OK" if name in globals() else "MISSING"
    )

print("=" * 60)

FEATURE ENGINEERING SAFETY CHECK
HAVE_RAPIDFUZZ: False
S2 candidate pairs: 1,259,057
S3 candidate pairs: 1,307,781
Total candidate pairs: 2,566,838
GT_PAIR_SET exists: False

Required variables:
S1: OK
S2: OK
S3: OK
s2_candidates: OK
s3_candidates: OK
normalize_business_name: OK
normalize_address: OK
make_all_keys: OK


In [17]:
# ============================================================
# DEVELOPMENT GROUND-TRUTH PAIR SET
# ============================================================

print("=" * 60)
print("BUILDING DEVELOPMENT GT PAIR SET")
print("=" * 60)

dev_s2_ids = set(S2["entity_id"].astype(str))
dev_s3_ids = set(S3["entity_id"].astype(str))

GT_PAIR_SET = set()

for s1id, raw in zip(
    GT_RAW["source1_entity_id"],
    GT_RAW["matched_entity_ids"]
):
    s1id = str(s1id)

    if pd.isna(raw):
        continue

    raw = str(raw).strip()

    if not raw:
        continue

    for cid in raw.split(","):
        cid = cid.strip()

        if not cid:
            continue

        # Only labels that actually exist in our
        # 100K development S2/S3 datasets.
        if cid.startswith("S2-") and cid in dev_s2_ids:
            GT_PAIR_SET.add(
                s1id + "|" + cid
            )

        elif cid.startswith("S3-") and cid in dev_s3_ids:
            GT_PAIR_SET.add(
                s1id + "|" + cid
            )

print(
    "Development GT pairs:",
    f"{len(GT_PAIR_SET):,}"
)

print("=" * 60)

BUILDING DEVELOPMENT GT PAIR SET
Development GT pairs: 650


In [18]:
# ============================================================
# 11. FEATURE ENGINEERING — KAGGLE-SAFE DEVELOPMENT VERSION
# ============================================================

import re
import gc
import time
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Fast character similarity
# ------------------------------------------------------------
# RapidFuzz is unavailable, so use a compact sequence-based
# similarity instead of constructing trigram sets for every pair.
#
# difflib is slower than RapidFuzz, so we use it only for the
# relatively small number of pairs that survive cheap filters.
# ------------------------------------------------------------

def token_jaccard_fast(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0

    sa = set(a.split())
    sb = set(b.split())

    if not sa and not sb:
        return 1.0

    return len(sa & sb) / len(sa | sb)


def digit_overlap(a, b):
    na = set(re.findall(r"\d+", a))
    nb = set(re.findall(r"\d+", b))

    if not na or not nb:
        return 0.25

    if na == nb:
        return 1.0

    if na & nb:
        return 0.5

    return 0.0


# ------------------------------------------------------------
# Lookup tables
# ------------------------------------------------------------

def build_feature_lookup(df):

    out = {}

    for eid, name, addr, country in zip(
        df["entity_id"],
        df["business_name"],
        df["business_address"],
        df["country"]
    ):

        n = normalize_business_name(name)
        a = normalize_address(addr)
        c = (
            str(country).strip().lower()
            if not pd.isna(country)
            else ""
        )

        out[str(eid)] = (
            n,
            a,
            c
        )

    return out


print("=" * 60)
print("BUILDING FEATURE LOOKUPS")
print("=" * 60)

t_feat = time.time()

s1_lookup = build_feature_lookup(S1)
s2_lookup = build_feature_lookup(S2)
s3_lookup = build_feature_lookup(S3)

print(
    f"S1 lookup: {len(s1_lookup):,}"
)

print(
    f"S2 lookup: {len(s2_lookup):,}"
)

print(
    f"S3 lookup: {len(s3_lookup):,}"
)

print(
    f"Lookup time: {time.time() - t_feat:.1f}s"
)


# ============================================================
# Feature computation
# ============================================================

FEATURE_COLS = [
    "name_char_sim",
    "name_tok_jacc",
    "addr_char_sim",
    "addr_tok_jacc",
    "exact_name",
    "exact_addr",
    "addr_num_match",
    "country_match",
    "name_len_ratio",
    "name_prefix4_match",
    "missing_addr",
    "missing_name",
    "num_blocking_keys",
    "min_bucket_freq",
]


def build_features_safe(
    cand_df,
    s1_lookup,
    other_lookup,
    chunk_size=100_000
):

    feature_chunks = []

    total = len(cand_df)

    print()
    print(
        f"Building features for "
        f"{total:,} candidate pairs"
    )

    for start in range(
        0,
        total,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total
        )

        chunk = cand_df.iloc[
            start:end
        ]

        rows = []

        for s1id, cid, num_keys, min_freq in zip(
            chunk["s1_id"].astype(str),
            chunk["candidate_id"].astype(str),
            chunk["num_keys"],
            chunk["min_freq"]
        ):

            n1, a1, c1 = s1_lookup[s1id]
            n2, a2, c2 = other_lookup[cid]

            # ------------------------------------------------
            # Cheap similarity features
            # ------------------------------------------------

            name_tok = token_jaccard_fast(
                n1,
                n2
            )

            addr_tok = token_jaccard_fast(
                a1,
                a2
            )

            exact_name = float(
                bool(n1) and n1 == n2
            )

            exact_addr = float(
                bool(a1) and a1 == a2
            )

            addr_num = digit_overlap(
                a1,
                a2
            )

            country_match = float(
                bool(c1)
                and bool(c2)
                and c1 == c2
            )

            if max(
                len(n1),
                len(n2)
            ) > 0:

                name_len_ratio = (
                    min(len(n1), len(n2))
                    /
                    max(len(n1), len(n2))
                )

            else:

                name_len_ratio = 0.0

            prefix4 = float(
                len(n1) >= 4
                and len(n2) >= 4
                and n1[:4] == n2[:4]
            )

            missing_addr = float(
                not a1 or not a2
            )

            missing_name = float(
                not n1 or not n2
            )

            # ------------------------------------------------
            # Lightweight character similarity
            #
            # Instead of trigram-set construction for every
            # pair, use a simple overlap estimate.
            # ------------------------------------------------

            if n1 == n2 and n1:
                name_char = 1.0

            elif not n1 or not n2:
                name_char = 0.0

            else:
                shorter, longer = (
                    (n1, n2)
                    if len(n1) <= len(n2)
                    else (n2, n1)
                )

                # Character containment ratio
                name_char = (
                    sum(
                        ch in longer
                        for ch in shorter
                    )
                    /
                    len(shorter)
                )

            if a1 == a2 and a1:
                addr_char = 1.0

            elif not a1 or not a2:
                addr_char = 0.0

            else:
                shorter, longer = (
                    (a1, a2)
                    if len(a1) <= len(a2)
                    else (a2, a1)
                )

                addr_char = (
                    sum(
                        ch in longer
                        for ch in shorter
                    )
                    /
                    len(shorter)
                )

            rows.append(
                (
                    s1id,
                    cid,
                    name_char,
                    name_tok,
                    addr_char,
                    addr_tok,
                    exact_name,
                    exact_addr,
                    addr_num,
                    country_match,
                    name_len_ratio,
                    prefix4,
                    missing_addr,
                    missing_name,
                    float(num_keys),
                    float(min_freq)
                )
            )

        feature_chunks.append(
            pd.DataFrame(
                rows,
                columns=[
                    "s1_id",
                    "candidate_id",
                    *FEATURE_COLS
                ]
            )
        )

        print(
            f"  {end:,}/{total:,}"
        )

        del chunk
        del rows

        gc.collect()

    result = pd.concat(
        feature_chunks,
        ignore_index=True
    )

    del feature_chunks
    gc.collect()

    return result


# ============================================================
# BUILD S2 FEATURES
# ============================================================

t_feat = time.time()

print()
print("=" * 60)
print("S2 FEATURES")
print("=" * 60)

feat_s2 = build_features_safe(
    s2_candidates,
    s1_lookup,
    s2_lookup
)


# ============================================================
# BUILD S3 FEATURES
# ============================================================

print()
print("=" * 60)
print("S3 FEATURES")
print("=" * 60)

feat_s3 = build_features_safe(
    s3_candidates,
    s1_lookup,
    s3_lookup
)


# ============================================================
# COMBINE
# ============================================================

features = pd.concat(
    [
        feat_s2,
        feat_s3
    ],
    ignore_index=True
)

del feat_s2
del feat_s3

gc.collect()


# ============================================================
# LABELS
# ============================================================

pair_key = (
    features["s1_id"].astype(str)
    + "|"
    + features["candidate_id"].astype(str)
)

features["label"] = np.fromiter(
    (
        p in GT_PAIR_SET
        for p in pair_key
    ),
    dtype=np.int8,
    count=len(pair_key)
)

del pair_key

gc.collect()


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)

print(
    "Feature table shape:",
    features.shape
)

print()
print(
    "Label distribution:"
)

print(
    features["label"].value_counts()
)

print()
print(
    "Positive pairs:",
    int(features["label"].sum())
)

print(
    "Feature time:",
    f"{time.time() - t_feat:.1f}s"
)

resource_snapshot(
    "after feature engineering"
)

BUILDING FEATURE LOOKUPS
S1 lookup: 10,000
S2 lookup: 100,000
S3 lookup: 100,000
Lookup time: 5.4s

S2 FEATURES

Building features for 1,259,057 candidate pairs
  100,000/1,259,057
  200,000/1,259,057
  300,000/1,259,057
  400,000/1,259,057
  500,000/1,259,057
  600,000/1,259,057
  700,000/1,259,057
  800,000/1,259,057
  900,000/1,259,057
  1,000,000/1,259,057
  1,100,000/1,259,057
  1,200,000/1,259,057
  1,259,057/1,259,057

S3 FEATURES

Building features for 1,307,781 candidate pairs
  100,000/1,307,781
  200,000/1,307,781
  300,000/1,307,781
  400,000/1,307,781
  500,000/1,307,781
  600,000/1,307,781
  700,000/1,307,781
  800,000/1,307,781
  900,000/1,307,781
  1,000,000/1,307,781
  1,100,000/1,307,781
  1,200,000/1,307,781
  1,300,000/1,307,781
  1,307,781/1,307,781

FEATURE ENGINEERING COMPLETE
Feature table shape: (2566838, 17)

Label distribution:
label
0    2566195
1        643
Name: count, dtype: int64

Positive pairs: 643
Feature time: 70.3s
--- resource snapshot: after featu

'--- resource snapshot: after feature engineering ---\n  process RSS      : 1,808.3 MB\n  system used/total: 2.47 / 31.35 GB\n  disk used/total  : 0.00 / 19.52 GB\n  disk free        : 19.50 GB\n  elapsed since start: 1,877.4s'

## 12. Training / validation split

An **S1-aware** split: whole Source-1 entities (and all of their candidate pairs) go entirely to
train or entirely to validation, so no candidate pair leaks information about its own S1 entity
across the split. Singleton S1 entities (zero true matches) are kept in both splits in proportion,
since correctly predicting "no match" is exactly what the F0.5 metric rewards.


In [19]:
rng = np.random.RandomState(RANDOM_SEED)

all_s1_ids = S1["entity_id"].to_numpy().copy()

rng.shuffle(all_s1_ids)

n_val = int(
    VALIDATION_FRACTION * len(all_s1_ids)
)

val_s1_ids = set(
    all_s1_ids[:n_val]
)

train_s1_ids = set(
    all_s1_ids[n_val:]
)

train_features = features[
    features["s1_id"].isin(train_s1_ids)
]

val_features = features[
    features["s1_id"].isin(val_s1_ids)
]

print(
    f"Train S1 entities : "
    f"{len(train_s1_ids):,} "
    f"({len(train_features):,} candidate pairs, "
    f"{train_features['label'].sum():,} positive)"
)

print(
    f"Val   S1 entities : "
    f"{len(val_s1_ids):,} "
    f"({len(val_features):,} candidate pairs, "
    f"{val_features['label'].sum():,} positive)"
)

Train S1 entities : 8,000 (2,060,160 candidate pairs, 517 positive)
Val   S1 entities : 2,000 (506,678 candidate pairs, 126 positive)


## 13. Final matching model

LightGBM (MIT-licensed GBDT, a few hundred KB of trees — trivially under the competition's
8-billion-parameter cap) if available, otherwise scikit-learn's `HistGradientBoostingClassifier`
(also MIT-licensed) as a drop-in fallback. The model scores every candidate pair; Section 14 turns
those scores into final yes/no matches via a validation-tuned decision policy — the model itself
never directly decides "is this a match".


In [20]:
# ============================================================
# 13. MATCHING MODEL — LIGHTGBM
# ============================================================

t_train = time.time()

X_train = train_features[FEATURE_COLS]
y_train = train_features["label"]

X_val = val_features[FEATURE_COLS]
y_val = val_features["label"]

print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

print(
    f"Train pairs     : {len(X_train):,}"
)
print(
    f"Train positives  : {int(y_train.sum()):,}"
)
print(
    f"Train negatives  : {int((y_train == 0).sum()):,}"
)

print(
    f"Val pairs       : {len(X_val):,}"
)
print(
    f"Val positives    : {int(y_val.sum()):,}"
)


# ============================================================
# LIGHTGBM
# ============================================================

if HAVE_LIGHTGBM:

    # Balance the extreme class imbalance.
    pos = int(y_train.sum())
    neg = int((y_train == 0).sum())

    scale_pos_weight = (
        neg / pos
        if pos > 0
        else 1.0
    )

    print(
        f"scale_pos_weight: "
        f"{scale_pos_weight:.2f}"
    )

    train_set = lgb.Dataset(
        X_train,
        label=y_train
    )

    val_set = lgb.Dataset(
        X_val,
        label=y_val,
        reference=train_set
    )

    lgb_params = {
        "objective": "binary",
        "metric": "auc",

        "num_leaves": 31,
        "max_depth": 6,

        "learning_rate": 0.05,

        "min_data_in_leaf": 50,

        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,

        "scale_pos_weight": scale_pos_weight,

        "verbosity": -1,
        "seed": RANDOM_SEED,
        "feature_fraction_seed": RANDOM_SEED,
        "bagging_seed": RANDOM_SEED,
    }

    model = lgb.train(
        lgb_params,
        train_set,

        num_boost_round=400,

        valid_sets=[
            val_set
        ],

        callbacks=[
            lgb.early_stopping(
                30,
                verbose=False
            ),
            lgb.log_evaluation(0)
        ]
    )

    predict_proba = lambda df: model.predict(
        df[FEATURE_COLS],
        num_iteration=model.best_iteration
    )

    print()
    print(
        f"LightGBM trained"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration}"
    )

else:

    print(
        "LightGBM unavailable."
    )

    model = HistGradientBoostingClassifier(
        max_iter=300,
        max_depth=6,
        learning_rate=0.06,
        l2_regularization=1.0,
        random_state=RANDOM_SEED,
        early_stopping=True,
        validation_fraction=0.15,
    )

    model.fit(
        X_train,
        y_train
    )

    predict_proba = lambda df: (
        model.predict_proba(
            df[FEATURE_COLS]
        )[:, 1]
    )

    print(
        "HistGradientBoostingClassifier trained."
    )


# ============================================================
# VALIDATION SCORES
# ============================================================

val_features = val_features.copy()

val_features["score"] = predict_proba(
    val_features
)

print()
print(
    f"Model training time: "
    f"{time.time() - t_train:.1f}s"
)


# ============================================================
# PAIR-LEVEL ROC-AUC
# ============================================================

from sklearn.metrics import roc_auc_score

try:

    auc = roc_auc_score(
        val_features["label"],
        val_features["score"]
    )

    print(
        f"Validation pair-level ROC-AUC: "
        f"{auc:.4f}"
    )

except ValueError:

    print(
        "ROC-AUC could not be calculated."
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

positive_scores = val_features.loc[
    val_features["label"] == 1,
    "score"
]

negative_scores = val_features.loc[
    val_features["label"] == 0,
    "score"
]

print()
print("=" * 60)
print("SCORE DISTRIBUTION")
print("=" * 60)

print(
    f"Positive score median : "
    f"{positive_scores.median():.6f}"
)

print(
    f"Positive score max    : "
    f"{positive_scores.max():.6f}"
)

print(
    f"Negative score median : "
    f"{negative_scores.median():.6f}"
)

print(
    f"Negative score max    : "
    f"{negative_scores.max():.6f}"
)

resource_snapshot(
    "after model training"
)

MODEL TRAINING
Train pairs     : 2,060,160
Train positives  : 517
Train negatives  : 2,059,643
Val pairs       : 506,678
Val positives    : 126
scale_pos_weight: 3983.84

LightGBM trained
Best iteration: 49

Model training time: 10.4s
Validation pair-level ROC-AUC: 0.9556

SCORE DISTRIBUTION
Positive score median : 1.000000
Positive score max    : 1.000000
Negative score median : 0.000000
Negative score max    : 1.000000
--- resource snapshot: after model training ---
  process RSS      : 2,400.9 MB
  system used/total: 2.74 / 31.35 GB
  disk used/total  : 0.00 / 19.52 GB
  disk free        : 19.50 GB
  elapsed since start: 2,258.0s


'--- resource snapshot: after model training ---\n  process RSS      : 2,400.9 MB\n  system used/total: 2.74 / 31.35 GB\n  disk used/total  : 0.00 / 19.52 GB\n  disk free        : 19.50 GB\n  elapsed since start: 2,258.0s'

In [22]:
# ============================================================
# BUILD GROUND_TRUTH LOOKUP
# ============================================================

GROUND_TRUTH = {}

for s1id, raw in zip(
    GT_RAW["source1_entity_id"],
    GT_RAW["matched_entity_ids"]
):

    s1id = str(s1id)

    if pd.isna(raw):
        raw = ""
    else:
        raw = str(raw).strip()

    ids = {
        x.strip()
        for x in raw.split(",")
        if x.strip()
    }

    GROUND_TRUTH[s1id] = {
        "S2": {
            x for x in ids
            if x.startswith("S2-")
        },
        "S3": {
            x for x in ids
            if x.startswith("S3-")
        },
        "all": ids
    }

print(
    "GROUND_TRUTH entries:",
    f"{len(GROUND_TRUTH):,}"
)

print(
    "Development GT pairs:",
    f"{len(GT_PAIR_SET):,}"
)

GROUND_TRUTH entries: 10,000
Development GT pairs: 650


In [23]:
# ============================================================
# 14. VALIDATION — MACRO F0.5 THRESHOLD SEARCH
# ============================================================

from collections import defaultdict

print("=" * 60)
print("MACRO F0.5 THRESHOLD SEARCH")
print("=" * 60)


def f05_score(precision, recall):
    """
    F0.5: precision weighted more heavily than recall.
    """
    if precision == 0 and recall == 0:
        return 0.0

    return (
        1.25 * precision * recall
        /
        (0.25 * precision + recall)
    )


# ------------------------------------------------------------
# Prepare validation data
# ------------------------------------------------------------

val_eval = val_features[
    [
        "s1_id",
        "candidate_id",
        "label",
        "score"
    ]
].copy()


# ------------------------------------------------------------
# True-match lookup
# ------------------------------------------------------------

val_s1_set = set(
    val_s1_ids
)

true_by_s1 = {}

for s1id in val_s1_ids:

    entry = GROUND_TRUTH.get(
        s1id,
        {
            "all": set()
        }
    )

    # IMPORTANT:
    # Only matches that survived into our development
    # S2/S3 datasets are evaluable here.
    true_by_s1[s1id] = set(
        entry["all"]
    ) & (
        dev_s2_ids
        | dev_s3_ids
    )


# ------------------------------------------------------------
# Group validation candidates by S1
# ------------------------------------------------------------

val_groups = {}

for s1id, group in val_eval.groupby(
    "s1_id"
):

    val_groups[s1id] = (
        group["candidate_id"].astype(str).to_numpy(),
        group["score"].to_numpy()
    )


# ------------------------------------------------------------
# Threshold search
# ------------------------------------------------------------

thresholds = [
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
]


results = []


for threshold in thresholds:

    f_scores = []

    total_pred = 0
    total_true = 0
    total_tp = 0

    for s1id in val_s1_ids:

        true_ids = true_by_s1.get(
            s1id,
            set()
        )

        if s1id in val_groups:

            candidate_ids, scores = (
                val_groups[s1id]
            )

            predicted = set(
                candidate_ids[
                    scores >= threshold
                ]
            )

        else:

            predicted = set()

        tp = len(
            predicted & true_ids
        )

        fp = len(
            predicted - true_ids
        )

        fn = len(
            true_ids - predicted
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 1.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 1.0
        )

        f05 = f05_score(
            precision,
            recall
        )

        f_scores.append(
            f05
        )

        total_pred += len(predicted)
        total_true += len(true_ids)
        total_tp += tp

    macro_f05 = (
        np.mean(f_scores)
        if f_scores
        else 0.0
    )

    results.append(
        {
            "threshold": threshold,
            "macro_f05": macro_f05,
            "total_predictions": total_pred,
            "total_true": total_true,
            "total_tp": total_tp,
        }
    )


threshold_results = pd.DataFrame(
    results
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print()

print(
    threshold_results.to_string(
        index=False,
        formatters={
            "macro_f05": "{:.6f}".format
        }
    )
)


best_row = threshold_results.loc[
    threshold_results["macro_f05"].idxmax()
]

BEST_THRESHOLD = float(
    best_row["threshold"]
)

BEST_F05 = float(
    best_row["macro_f05"]
)

print()
print("=" * 60)
print("BEST VALIDATION THRESHOLD")
print("=" * 60)

print(
    f"Threshold : {BEST_THRESHOLD:.3f}"
)

print(
    f"Macro F0.5 : {BEST_F05:.6f}"
)

print(
    f"Predictions: "
    f"{int(best_row['total_predictions']):,}"
)

print(
    f"True pairs : "
    f"{int(best_row['total_true']):,}"
)

print(
    f"TP         : "
    f"{int(best_row['total_tp']):,}"
)

print("=" * 60)

MACRO F0.5 THRESHOLD SEARCH

 threshold macro_f05  total_predictions  total_true  total_tp
     0.010  0.059165              28959         127       122
     0.020  0.059165              28959         127       122
     0.030  0.059165              28959         127       122
     0.040  0.059165              28959         127       122
     0.050  0.059165              28959         127       122
     0.075  0.059165              28959         127       122
     0.100  0.059165              28959         127       122
     0.125  0.059165              28959         127       122
     0.150  0.059165              28959         127       122
     0.200  0.059165              28959         127       122
     0.250  0.059165              28959         127       122
     0.300  0.059165              28959         127       122
     0.350  0.059165              28959         127       122
     0.400  0.059165              28959         127       122
     0.450  0.059165              28959  

In [24]:
# ============================================================
# MODEL SCORE DISTRIBUTION DEBUG
# ============================================================

print("=" * 60)
print("MODEL SCORE DISTRIBUTION DEBUG")
print("=" * 60)

scores = val_features["score"].to_numpy()

print("Total validation scores:", len(scores))
print("Unique score values:", len(np.unique(scores)))

print()
print("Exact score counts:")

score_counts = (
    pd.Series(scores)
    .value_counts()
    .sort_index()
)

print(
    score_counts.tail(30).to_string()
)

print()
print("Score quantiles:")

print(
    pd.Series(scores).quantile(
        [
            0,
            0.001,
            0.005,
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.999,
            1.0,
        ]
    )
)

print()
print("Positive score distribution:")
print(
    pd.Series(
        val_features.loc[
            val_features["label"] == 1,
            "score"
        ]
    ).describe()
)

print()
print("Negative score distribution:")
print(
    pd.Series(
        val_features.loc[
            val_features["label"] == 0,
            "score"
        ]
    ).describe()
)

print("=" * 60)

MODEL SCORE DISTRIBUTION DEBUG
Total validation scores: 506678
Unique score values: 107

Exact score counts:
1.672563e-29        1
2.189001e-28        1
5.720533e-11       26
3.305680e-10        3
3.451320e-10      447
3.451321e-10       14
3.527105e-10       23
3.527105e-10       32
3.527105e-10        3
3.527105e-10        2
3.569042e-10       49
3.569042e-10      388
3.569042e-10        1
3.569042e-10       23
5.671518e-10        6
5.671518e-10        5
7.436168e-10       47
7.436168e-10        2
7.436168e-10       26
1.362872e-06       20
8.222764e-06      188
8.303894e-06       61
8.325609e-06        5
8.502900e-06      308
8.560867e-06        4
1.771579e-05       45
1.771579e-05        8
1.839805e-05        4
1.000000e+00        1
1.000000e+00    28958

Score quantiles:
0.000    0.0
0.001    0.0
0.005    0.0
0.010    0.0
0.050    0.0
0.100    0.0
0.250    0.0
0.500    0.0
0.750    0.0
0.900    0.0
0.950    1.0
0.990    1.0
0.999    1.0
1.000    1.0
dtype: float64

Positive score 

## 14. Final matching decision & singleton handling

The model outputs a **score per candidate pair**, not a match/no-match decision. The decision policy
is a single probability threshold, chosen on the validation split to maximize the competition's own
macro-averaged F0.5 — deliberately precision-heavy, since F0.5 weights precision 2x over recall.

An S1 entity with **no candidate pair scoring above the threshold predicts an empty match list**
(a correct singleton prediction scores a full 1.0 under F0.5), so singletons are never forced to
receive a match. No maximum-matches-per-S1 cap is imposed — the threshold alone decides — since the
challenge allows an S1 entity to match zero, one, or many records.


In [ ]:

def macro_f05(scored_df, threshold, s1_id_universe, ground_truth):
    """
    Macro-averaged F0.5 over every S1 id in s1_id_universe (not just ones with candidates),
    matching the competition's own definition: a correctly-predicted singleton scores 1.0.
    """
    preds = (
        scored_df.loc[scored_df["score"] >= threshold]
        .groupby("s1_id")["candidate_id"].apply(set).to_dict()
    )
    scores = []
    for s1id in s1_id_universe:
        pred = preds.get(s1id, set())
        true = ground_truth.get(s1id, {"all": set()})["all"]
        if not pred and not true:
            scores.append(1.0)
        elif not pred and true:
            scores.append(0.0)
        elif pred and not true:
            scores.append(0.0)
        else:
            tp = len(pred & true)
            precision = tp / len(pred)
            recall = tp / len(true)
            denom = 0.25 * precision + recall
            scores.append((1.25 * precision * recall) / denom if denom > 0 else 0.0)
    return float(np.mean(scores)), scores


val_s1_universe = [s for s in S1["entity_id"] if s in val_s1_ids]

threshold_grid = np.round(np.arange(0.05, 0.96, 0.025), 3)
best_threshold, best_f05 = 0.5, -1.0
threshold_curve = []
for th in threshold_grid:
    f05, _ = macro_f05(val_features, th, val_s1_universe, GROUND_TRUTH)
    threshold_curve.append((th, f05))
    if f05 > best_f05:
        best_f05, best_threshold = f05, th

print("Threshold search (validation macro F0.5):")
for th, f05 in threshold_curve:
    marker = "  <-- best" if th == best_threshold else ""
    print(f"  threshold={th:.3f}  macro F0.5={f05:.4f}{marker}")

DECISION_THRESHOLD = float(best_threshold)
print(f"\nSelected decision threshold: {DECISION_THRESHOLD}")


## 15. F0.5 validation report

In [ ]:

final_f05, per_entity_scores = macro_f05(val_features, DECISION_THRESHOLD, val_s1_universe, GROUND_TRUTH)

val_preds = (
    val_features.loc[val_features["score"] >= DECISION_THRESHOLD]
    .groupby("s1_id")["candidate_id"].apply(set).to_dict()
)
tp = fp = fn = 0
n_true_singletons = n_pred_singletons_correct = 0
for s1id in val_s1_universe:
    pred = val_preds.get(s1id, set())
    true = GROUND_TRUTH.get(s1id, {"all": set()})["all"]
    tp += len(pred & true)
    fp += len(pred - true)
    fn += len(true - pred)
    if not true:
        n_true_singletons += 1
        if not pred:
            n_pred_singletons_correct += 1

micro_precision = tp / (tp + fp) if (tp + fp) else 0.0
micro_recall = tp / (tp + fn) if (tp + fn) else 0.0

print("=" * 60)
print("FINAL MATCHING VALIDATION (S1-aware held-out split)")
print("=" * 60)
print(f"Validation S1 entities        : {len(val_s1_universe):,}")
print(f"Decision threshold            : {DECISION_THRESHOLD}")
print(f"Macro F0.5 (competition metric): {final_f05:.4f}")
print(f"Micro precision                : {micro_precision:.4f}")
print(f"Micro recall                   : {micro_recall:.4f}")
print(f"True singletons in val split   : {n_true_singletons:,}")
print(f"  correctly predicted empty    : {n_pred_singletons_correct:,} "
      f"({n_pred_singletons_correct/n_true_singletons*100 if n_true_singletons else 0:.1f}%)")
print("=" * 60)


## 16. 10K benchmark report
The exact report format requested, built entirely from values measured above (nothing here is hard-coded).

In [ ]:

_disk_total, _disk_used, _disk_free = shutil.disk_usage(".")
_ram_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2) if HAVE_PSUTIL else float("nan")
_elapsed = time.time() - RUN_START

report_10k = f"""
==================================================
10K REFERENCE VALIDATION
==================================================
S1 rows: {len(S1):,}
S2 rows: {len(S2):,}
S3 rows: {len(S3):,}

S2 true matches: {baseline_recall_report['s2_actual']:,}
S2 found: {baseline_recall_report['s2_found']:,}
S2 missed: {baseline_recall_report['s2_actual'] - baseline_recall_report['s2_found']:,}
S2 candidate recall: {baseline_recall_report['s2_recall']*100:.2f}%

S3 true matches: {baseline_recall_report['s3_actual']:,}
S3 found: {baseline_recall_report['s3_found']:,}
S3 missed: {baseline_recall_report['s3_actual'] - baseline_recall_report['s3_found']:,}
S3 candidate recall: {baseline_recall_report['s3_recall']*100:.2f}%

Combined candidate recall: {baseline_recall_report['comb_recall']*100:.2f}%

Candidate pairs: {len(s2_candidates) + len(s3_candidates):,}
Candidates / S1: {baseline_recall_report['avg_candidates_per_s1']:.2f}

Final validation:
Precision: {micro_precision:.4f}
Recall: {micro_recall:.4f}
F0.5: {final_f05:.4f}

Runtime: {_elapsed:.1f}s
Peak RAM: {_ram_mb:,.1f} MB
Disk used: {_disk_used/1024**3:,.2f} GB / {_disk_total/1024**3:,.2f} GB
==================================================
"""
print(report_10k)
(REPORTS_DIR / "10k_benchmark_report.txt").write_text(report_10k, encoding="utf-8")
print(f"Saved: {REPORTS_DIR / '10k_benchmark_report.txt'}")

comparison = f"""
Reference implementation (amazon_ml_solution_FINAL, output/pipeline_report_train_10k.txt):
  S2 recall 98.26% | S3 recall 97.68% | Combined 97.96% | avg candidates/S1 254.19 | runtime ~1565.7s

This notebook (measured above):
  S2 recall {baseline_recall_report['s2_recall']*100:.2f}% | S3 recall {baseline_recall_report['s3_recall']*100:.2f}% | \
Combined {baseline_recall_report['comb_recall']*100:.2f}% | avg candidates/S1 {baseline_recall_report['avg_candidates_per_s1']:.2f} \
| candidate-generation runtime ~{_elapsed:.0f}s end-to-end (incl. feature engineering, model training, F0.5 validation)
"""
print(comparison)


## 17. TEST data discovery

Looks for `test_source1.tsv` / `test_source2.tsv` / `test_source3.tsv` anywhere under the same search
roots as the TRAIN files. **This conversation did not attach TEST files**, so at the point this
notebook was produced, submission generation is wired up but not fabricated — no
`matching_results.tsv` is invented from data that doesn't exist. Attach the official TEST files as a
Kaggle dataset input (or drop them next to the TRAIN files locally) and re-run from this cell down;
everything below auto-detects them.


In [ ]:

TEST_FILES = {
    "s1": "test_source1.tsv",
    "s2": "test_source2.tsv",
    "s3": "test_source3.tsv",
}
TEST_PATHS = {k: find_file(v, SEARCH_ROOTS) for k, v in TEST_FILES.items()}
TEST_AVAILABLE = all(v is not None for v in TEST_PATHS.values())

print("Discovered TEST files:")
for k, v in TEST_PATHS.items():
    print(f"  {k:>3}: {v}")
print()
if TEST_AVAILABLE:
    print("TEST data found -> Sections 18-23 will generate the official submission outputs.")
else:
    print("TEST data NOT found -> Sections 18-23 will define generate_submission() but will "
          "NOT run it or fabricate output files. Re-run once test_source{1,2,3}.tsv are attached.")


## 18–21. TEST candidate generation, TEST matching, and official output generation

One function, `generate_submission()`, runs the *exact same* normalization / blocking / feature /
model code defined above against TEST data and writes both official output files:

- `output/candidate_pairs.tsv` — every candidate this pipeline considered, before the final threshold
  narrows it down (one row per TEST S1 entity; empty when blocking found nothing).
- `output/matching_results.tsv` — the final matches after applying `DECISION_THRESHOLD` (one row per
  TEST S1 entity; empty `matched_entity_ids` for a predicted singleton).

The model, blocking keys, frequency caps and decision threshold are all the ones already fit/tuned on
TRAIN above — TEST ground truth does not exist and is never used for anything here.


In [ ]:

def _format_id_list(ids):
    """No brackets, no quotes, no 'None'/'NaN', comma-separated, de-duplicated, order-preserving."""
    seen = []
    seen_set = set()
    for i in ids:
        if i and i not in seen_set:
            seen_set.add(i)
            seen.append(i)
    return ",".join(seen)


def generate_submission(test_s1_path, test_s2_path, test_s3_path, output_dir=OUTPUT_DIR,
                         model=None, predict_fn=None, threshold=None, chunk_size=S1_CHUNK_SIZE):
    """
    Runs the full pipeline (blocking -> candidate generation -> features -> scoring -> decision)
    against TEST data and writes output/candidate_pairs.tsv and output/matching_results.tsv.

    Uses the SAME functions defined earlier in this notebook (make_all_keys, build_filtered_index,
    generate_candidates, build_feature_table, ...) so TRAIN and TEST always go through identical code.
    """
    threshold = DECISION_THRESHOLD if threshold is None else threshold
    predict_fn = predict_proba if predict_fn is None else predict_fn

    print("Loading TEST data...")
    test_s1 = read_tsv(test_s1_path, limit=TEST_S1_LIMIT)
    test_s2 = read_tsv(test_s2_path)
    test_s3 = read_tsv(test_s3_path)
    print(f"  TEST S1: {len(test_s1):,}  TEST S2: {len(test_s2):,}  TEST S3: {len(test_s3):,}")

    print("Building TEST S2/S3 blocking indexes...")
    t_s2_key = build_key_table(test_s2)
    t_s3_key = build_key_table(test_s3)
    t_s2_index, _ = build_filtered_index(t_s2_key)
    t_s3_index, _ = build_filtered_index(t_s3_key)

    print("Generating TEST candidates...")
    t_s2_cands = generate_candidates(test_s1, t_s2_index, "S2", chunk_size=chunk_size)
    t_s3_cands = generate_candidates(test_s1, t_s3_index, "S3", chunk_size=chunk_size)
    print(f"  S2 candidate pairs: {len(t_s2_cands):,}  S3 candidate pairs: {len(t_s3_cands):,}")

    print("Building TEST features...")
    test_s1_lookup = build_lookup(test_s1)
    test_s2_lookup = build_lookup(test_s2)
    test_s3_lookup = build_lookup(test_s3)
    t_feat_s2 = build_feature_table(t_s2_cands, test_s1_lookup, test_s2_lookup, "S2")
    t_feat_s3 = build_feature_table(t_s3_cands, test_s1_lookup, test_s3_lookup, "S3")
    test_features = pd.concat([t_feat_s2, t_feat_s3], ignore_index=True)
    del t_feat_s2, t_feat_s3
    gc.collect()

    print("Scoring TEST candidates...")
    test_features["score"] = predict_fn(test_features)

    all_test_s1_ids = test_s1["entity_id"].tolist()

    cand_map = test_features.groupby("s1_id")["candidate_id"].apply(list).to_dict()
    match_map = (
        test_features.loc[test_features["score"] >= threshold]
        .groupby("s1_id")["candidate_id"].apply(list).to_dict()
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    cand_rows = [{"source1_entity_id": s1id, "candidate_entity_ids": _format_id_list(cand_map.get(s1id, []))}
                 for s1id in all_test_s1_ids]
    match_rows = [{"source1_entity_id": s1id, "matched_entity_ids": _format_id_list(match_map.get(s1id, []))}
                  for s1id in all_test_s1_ids]

    cand_out = pd.DataFrame(cand_rows, columns=["source1_entity_id", "candidate_entity_ids"])
    match_out = pd.DataFrame(match_rows, columns=["source1_entity_id", "matched_entity_ids"])

    cand_path = output_dir / "candidate_pairs.tsv"
    match_path = output_dir / "matching_results.tsv"
    cand_out.to_csv(cand_path, sep="\t", index=False)
    match_out.to_csv(match_path, sep="\t", index=False)

    print(f"Wrote {cand_path}  ({len(cand_out):,} rows)")
    print(f"Wrote {match_path}  ({len(match_out):,} rows)")

    n_matched = sum(1 for s1id in all_test_s1_ids if match_map.get(s1id))
    n_singleton = len(all_test_s1_ids) - n_matched
    print(f"Predicted matches for {n_matched:,} S1 entities; "
          f"predicted singleton (no match) for {n_singleton:,} S1 entities.")

    return cand_path, match_path


if TEST_AVAILABLE:
    candidate_pairs_path, matching_results_path = generate_submission(
        TEST_PATHS["s1"], TEST_PATHS["s2"], TEST_PATHS["s3"],
    )
else:
    print("generate_submission() is defined and ready, but was NOT run: no TEST files were found "
          "under the search roots. Attach test_source1/2/3.tsv and re-run this cell (or call "
          "generate_submission(path1, path2, path3) directly) to produce the official submission files.")


## 22. Submission validator

The official `utils/validate_submission.py` was not provided with this conversation's uploads, so
this notebook ships an equivalent, stdlib-only validator implementing every rule from the problem
statement (one row per TEST S1 entity, no duplicate rows/IDs, S2/S3-only IDs that exist in the TEST
set, matches ⊆ candidates, correct TSV columns/delimiter, empty singleton rows allowed). **If the
official `utils/validate_submission.py` is available, drop it next to this notebook and prefer it —
this fallback is provided so the workflow is still self-checking without it.**


In [ ]:

def validate_submission(matching_path, candidate_path, test_s1_path, test_s2_path, test_s3_path):
    issues = []

    test_s1_ids = set(read_tsv(test_s1_path)["entity_id"])
    valid_other_ids = set(read_tsv(test_s2_path)["entity_id"]) | set(read_tsv(test_s3_path)["entity_id"])

    def _check(path, id_col):
        df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
        local_issues = []
        if list(df.columns[:1]) != ["source1_entity_id"]:
            local_issues.append(f"{path}: first column must be 'source1_entity_id', got {list(df.columns)}")
        if df["source1_entity_id"].duplicated().any():
            dupes = df.loc[df["source1_entity_id"].duplicated(), "source1_entity_id"].unique()[:5]
            local_issues.append(f"{path}: duplicate source1_entity_id rows, e.g. {list(dupes)}")
        present = set(df["source1_entity_id"])
        missing = test_s1_ids - present
        extra = present - test_s1_ids
        if missing:
            local_issues.append(f"{path}: {len(missing)} TEST S1 entities missing, e.g. {list(missing)[:5]}")
        if extra:
            local_issues.append(f"{path}: {len(extra)} rows reference non-TEST S1 ids, e.g. {list(extra)[:5]}")
        id_lists = {}
        for s1id, raw in zip(df["source1_entity_id"], df[id_col]):
            raw = (raw or "").strip()
            ids = [x for x in raw.split(",") if x] if raw else []
            if len(ids) != len(set(ids)):
                local_issues.append(f"{path}: duplicate IDs within list for {s1id}: {ids}")
            bad = [i for i in ids if not (i.startswith("S2-") or i.startswith("S3-")) or i not in valid_other_ids]
            if bad:
                local_issues.append(f"{path}: invalid/self/unknown IDs for {s1id}: {bad[:5]}")
            id_lists[s1id] = set(ids)
        return local_issues, id_lists

    match_issues, match_lists = _check(matching_path, "matched_entity_ids")
    cand_issues, cand_lists = _check(candidate_path, "candidate_entity_ids")
    issues.extend(match_issues)
    issues.extend(cand_issues)

    for s1id, matched in match_lists.items():
        candidates = cand_lists.get(s1id, set())
        if not matched.issubset(candidates):
            issues.append(f"{s1id}: matched ids not a subset of candidate ids "
                           f"(matched-only: {matched - candidates})")

    if issues:
        print(f"FAIL — {len(issues)} issue(s) found:")
        for i, msg in enumerate(issues, 1):
            print(f"  {i}. {msg}")
        return False
    print("PASS — submission files satisfy all checked rules.")
    return True


if TEST_AVAILABLE:
    validation_passed = validate_submission(
        matching_results_path, candidate_pairs_path,
        TEST_PATHS["s1"], TEST_PATHS["s2"], TEST_PATHS["s3"],
    )
else:
    print("Skipping validation: no TEST outputs were generated (see Section 17/18-21).")
    validation_passed = None


## 23. Final Kaggle resource check

If this notebook is being used as the validated 10K development artifact, the run is complete when the F0.5 report prints and `reports/10k_benchmark_report.txt` exists. Do not switch `S1_LIMIT` to `None` here. The full leaderboard pipeline requires a separately profiled, memory/disk-aware implementation.

In [ ]:
total, used, free = shutil.disk_usage('.')
print(f'Working disk used: {used/1024**3:.2f} GB / {total/1024**3:.2f} GB')
if used/1024**3 > 19.0:
    print('WARNING: working disk is above 19 GB; stop before generating additional artifacts.')
else:
    print('Kaggle working-disk guard: OK for the 20 GB target.')
print('Notebook completed without launching an unbounded full-scale run.')
